# 🚀 GLIR-HyperGCN: All-in-One Google Colab & Local Notebook (NTU60 Cross-Subject / 60CSub)
> **Mô hình**: **GLIR-HyperGCN (Grounded Logic Iterative Refinement với backbone HyperGCN)**
> **Nâng cấp từ**: `CLIR-SkateFormer` — xem `Architecture/10_CLIR-SkateFormer_v2_Proposal.md`
> **Benchmark mặc định**: **NTU RGB+D 60 Cross-Subject (60CSub / NTU60_CS)**
> **Số lớp hành động**: **60 classes** (A1–A60)
> **Số khái niệm**: **72 concepts** (52 spatial + 20 temporal), backbone **HyperGCN (NTU-25 chuẩn)**
> **3 khoảng trống đã vá so với CLIR-SkateFormer gốc**:
> 1. **G1 — BodyPartGroundedConceptHeads**: concept không gian được dự đoán RIÊNG theo từng nhóm khớp NTU-25 (Head/Hand/Arm/Hip/Leg/Foot), khớp 1-1 với part-grouped L1 ở tầng logic (trước đây dùng chung 1 vector pooled toàn cục cho mọi concept).
> 2. **G2 — LightConceptGrounding**: CLIP đóng băng (không LoRA), text anchor encode 1 lần, chỉ học 1 projection nhỏ.
> 3. **G3 — set_progress()**: annealing liên tục (temperature/beta_max/lambda_reg) thay `set_phase()` đột ngột — chọn qua biến `VARIANT` (`v2` = set_phase cũ, `v3` = set_progress mới).

---

### 📌 Cấu trúc Toàn diện Notebook:
1. **Block 1**: Môi trường, Dependencies (torch, CLIP) & Mount Google Drive
2. **Block 2**: Concept Ontology CSV & NTU Feeder
3. **Block 3**: Perception Backbone HyperGCN
4. **Block 4**: Tầng Logic CRL (LiteralLayer, ConjunctionLayer, DisjunctionLayer, CRLReasoner)
5. **Block 5**: G2 — LightConceptGrounding (CLIP đóng băng)
6. **Block 6**: ConceptUpdateCell, IterativeLogicReasoner, BodyPartGroundedConceptHeads & GLIRHyperGCN
7. **Block 7**: RuleExtractor (Post-hoc DNF Rule Extraction & Fidelity Engine)
8. **Block 8**: Kiểm thử nhanh tính toàn vẹn (Quick Smoke Test — cả V2 và V3)
9. **Block 9**: Thiết lập Cấu hình, chọn VARIANT (v2/v3) & DataLoaders
10. **Block 10**: Vòng lặp Huấn luyện (GLIRTrainer) & Lưu Checkpoint
11. **Block 11**: Trích xuất & Đánh giá Luật DNF (Precision, Coverage, Fidelity)
12. **Block 12**: Huấn luyện & Trích xuất qua Dòng lệnh CLI (train.py / scripts/extract_rules.py)



## 📦 Block 1: Cài đặt Dependencies & Môi trường


In [ ]:
# Cài đặt thư viện cần thiết
!pip install timm tensorboardX pyyaml tqdm ftfy regex -q
!pip install git+https://github.com/openai/CLIP.git -q

import os
import sys
import time
import json
import torch
import numpy as np

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

# Mount Google Drive nếu chạy trên Google Colab
if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')



## 📑 Block 2: Concept Ontology & Data Feeder


In [ ]:
# Lưu concept ontology CSV vào workspace (72 concepts chuẩn hóa)
os.makedirs("concepts", exist_ok=True)
with open("concepts/ntu_rgbd_60_120.csv", "w", encoding="utf-8") as f:
    f.write('''action_class,head_nod,head_shake,head_tilt_down,head_tilt_up,head_rotate,head_static,hand_grasp,hand_release,hand_wave,hand_point,hand_clap,hand_manipulate,hand_fist,hand_open,hand_touch,hand_lift_to_face,hand_circular,arm_raise,arm_lower,arm_extend,arm_retract,arm_circular,arm_swing,arm_cross,arm_hold,arm_alternating,arm_overhead,arm_lateral,hip_bend,hip_extend,hip_sit,hip_squat,hip_rotate,hip_static_stand,hip_static_sit,hip_sway,leg_step,leg_kick,leg_jump,leg_squat,leg_lift,leg_static_stand,leg_static_sit,leg_alternating,leg_balance,foot_step,foot_kick,foot_jump,foot_pivot,foot_static,foot_slide,interaction_two_person,temporal_direction_motion_upward,temporal_direction_motion_downward,temporal_direction_motion_forward,temporal_direction_motion_backward,temporal_direction_motion_converging,temporal_direction_motion_diverging,temporal_direction_motion_reversible,temporal_sequence_seq_hands_first,temporal_sequence_seq_legs_first,temporal_sequence_seq_body_first,temporal_sequence_seq_simultaneous,temporal_sequence_seq_alternating,temporal_dynamics_speed_slow,temporal_dynamics_speed_fast,temporal_dynamics_speed_accelerating,temporal_dynamics_speed_decelerating,temporal_dynamics_rhythm_regular,temporal_dynamics_rhythm_irregular,temporal_dynamics_duration_brief,temporal_dynamics_duration_sustained
drink_water,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0
eat_meal,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1
brush_teeth,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,1
brush_hair,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1
drop,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0
pick_up,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0
throw,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0
sit_down,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0
stand_up,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0
clapping,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,1,0,0,1,0,0,1
reading,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1
writing,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,1
tear_up_paper,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,1,0
put_on_jacket,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1
take_off_jacket,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1
put_on_a_shoe,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0
take_off_a_shoe,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0
put_on_glasses,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0
take_off_glasses,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0
put_on_a_hat_cap,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0
take_off_a_hat_cap,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0
cheer_up,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0
hand_waving,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1
kicking_something,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0
reach_into_pocket,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
hopping,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1
jump_up,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0
phone_call,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1
play_with_phone_tablet,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1
type_on_a_keyboard,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,1
point_to_something,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
taking_a_selfie,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
check_time_(from_watch),0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
rub_two_hands,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1
nod_head_bow,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0
shake_head,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0
wipe_face,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0
salute,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
put_palms_together,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0
cross_hands_in_front,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0
sneeze_cough,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0
staggering,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,1
falling_down,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0
headache,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1
chest_pain,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1
back_pain,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1
neck_pain,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1
nausea_vomiting,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0
fan_self,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,1
punch_slap,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0
kicking,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0
pushing,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0
pat_on_back,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1
point_finger,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
hugging,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1
giving_object,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
touch_pocket,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
shaking_hands,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0
walking_towards,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1
walking_apart,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1
put_on_headphone,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0
take_off_headphone,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,1,0
shoot_at_basket,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0
bounce_ball,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1
tennis_bat_swing,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0
juggle_table_tennis_ball,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,1
hush,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
flick_hair,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
thumb_up,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
thumb_down,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
make_OK_sign,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
make_victory_sign,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
staple_book,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
counting_money,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1
cutting_nails,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,1
cutting_paper,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1
snap_fingers,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
open_bottle,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,1,0
sniff_smell,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0
squat_down,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0
toss_a_coin,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
fold_paper,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1
ball_up_paper,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0
play_magic_cube,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,1
apply_cream_on_face,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1
apply_cream_on_hand,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1
put_on_bag,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0
take_off_bag,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0
put_object_into_bag,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
take_object_out_of_bag,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
open_a_box,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,1,0
move_heavy_objects,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1
shake_fist,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,1,0
throw_up_cap_hat,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
capitulate,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0
cross_arms,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1
arm_circles,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,1
arm_swings,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,1
run_on_the_spot,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,1
butt_kicks,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,1
cross_toe_touch,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,1
side_kick,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0
yawn,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1
stretch_oneself,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1
blow_nose,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
hit_with_object,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0
wield_knife,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
knock_over,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
grab_stuff,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
shoot_with_gun,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
step_on_foot,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0
high-five,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0
cheers_and_drink,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0
carry_object,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1
take_a_photo,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
follow,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1
whisper,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1
exchange_things,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0
support_somebody,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1
rock-paper-scissors,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,1,0
''')
print("Concept CSV written: concepts/ntu_rgbd_60_120.csv (72 concepts)")



In [ ]:
# Data Augmentation Tools
import os
import sys
import types

# 1. Lưu feeders/tools.py vào workspace để DataLoader workers có thể import feeders.tools bình thường
os.makedirs("feeders", exist_ok=True)
with open("feeders/__init__.py", "w", encoding="utf-8") as f:
    f.write("")
with open("feeders/tools.py", "w", encoding="utf-8") as f:
    f.write("import random\nimport numpy as np\nimport math\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\ndef valid_crop_resize(data_numpy, valid_frame_num, p_interval, window, thres):\n    # input: C,T,V,M\n    if isinstance(p_interval, (int, float)):\n        p_interval = [p_interval]\n    C, T, V, M = data_numpy.shape\n    begin = 0\n    end = valid_frame_num if valid_frame_num > 0 else T\n    valid_size = max(1, end - begin)\n\n    # crop\n    if len(p_interval) == 1:\n        p = p_interval[0]\n        bias = int((1 - p) * valid_size / 2)\n        c_b = min(begin + bias, T - 1)\n        c_e = max(c_b + 1, min(end - bias, T))\n        data = data_numpy[:, c_b:c_e, :, :]  # center_crop\n        cropped_length = data.shape[1]\n    else:\n        p = np.random.rand(1) * (p_interval[1] - p_interval[0]) + p_interval[0]\n        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres), valid_size)  # constraint cropped_length lower bound as thres\n        max_bias = max(0, valid_size - cropped_length)\n        bias = np.random.randint(0, max_bias + 1) if max_bias > 0 else 0\n        c_b = begin + bias\n        c_e = min(begin + bias + cropped_length, T)\n        data = data_numpy[:, c_b:c_e, :, :]\n        if data.shape[1] == 0:\n            print(cropped_length, bias, valid_size)\n\n    # resize\n    data = torch.tensor(data, dtype=torch.float)  # C, crop_t, V, M\n    data = data.permute(2, 3, 0, 1).contiguous().view(V * M, C, cropped_length)  # V*M, C, crop_t\n    data = F.interpolate(data, size=window, mode='linear', align_corners=False)  # V*M, C, T\n    data = data.contiguous().view(V, M, C, window).permute(2, 3, 0, 1).contiguous().numpy()\n    index_t = torch.arange(start=c_b, end=c_e, dtype=torch.float)\n    index_t = F.interpolate(index_t[None, None, :], size=window, mode='linear', align_corners=False).squeeze()\n    index_t = 2 * index_t / valid_size - 1\n    return data, index_t.numpy()\n\n\ndef valid_crop_uniform(data_numpy, valid_frame_num, p_interval, window, thres):\n    # input: C,T,V,M\n    if isinstance(p_interval, (int, float)):\n        p_interval = [p_interval]\n    C, T, V, M = data_numpy.shape\n    begin = 0\n    end = valid_frame_num if valid_frame_num > 0 else T\n    valid_size = max(1, end - begin)\n\n    # crop\n    if len(p_interval) == 1:\n        p = p_interval[0]\n        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres), valid_size)\n        bias = int((1 - p) * valid_size / 2)\n        bias = min(bias, max(0, valid_size - cropped_length))\n\n        if cropped_length < window:\n            inds = np.arange(cropped_length)\n        else:\n            bids = np.array(\n                [i * cropped_length // window for i in range(window + 1)])\n            bst = bids[:window]\n            inds = bst\n\n        inds = inds + bias\n        inds = np.clip(inds, 0, T - 1)\n        data = data_numpy[:, inds, :, :]\n\n    else:\n        p = np.random.rand(1) * (p_interval[1] - p_interval[0]) + p_interval[0]\n        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres),\n                                    valid_size)  # constraint cropped_length lower bound as 64\n        max_bias = max(0, valid_size - cropped_length)\n        bias = np.random.randint(0, max_bias + 1) if max_bias > 0 else 0\n\n        if cropped_length < window:\n            inds = np.arange(cropped_length)\n        elif window <= cropped_length < 2 * window:\n            basic = np.arange(window)\n            inds = np.random.choice(window + 1, cropped_length - window, replace=False)\n            offset = np.zeros(window + 1, dtype=np.int64)\n            offset[inds] = 1\n            offset = np.cumsum(offset)\n            inds = basic + offset[:-1]\n        else:\n            bids = np.array([i * cropped_length // window for i in range(window + 1)])\n            bsize = np.diff(bids)\n            bst = bids[:window]\n            offset = np.random.randint(bsize)\n            inds = bst + offset\n\n        inds = inds + bias\n        inds = np.clip(inds, 0, T - 1)\n        data = data_numpy[:, inds, :, :]\n        if data.shape[1] == 0:\n            print(cropped_length, bias, valid_size)\n\n    # resize\n    data = torch.tensor(data, dtype=torch.float)\n    index_t = torch.tensor(inds, dtype=torch.float)\n    data = data.permute(2, 3, 0, 1).contiguous().view(V * M, C, len(inds))  # V*M, C, crop_t\n\n    if len(inds) != window:\n        data = F.interpolate(data, size=window, mode='linear', align_corners=False)  # V*M, C, T\n        index_t = F.interpolate(index_t[None, None, :], size=window, mode='linear', align_corners=False).squeeze()\n\n    data = data.contiguous().view(V, M, C, window).permute(2, 3, 0, 1).contiguous().numpy()\n    index_t = 2 * index_t / valid_size - 1\n    return data, index_t.numpy()\n\n\ndef scale(data_numpy, scale=0.2, p=0.5):\n    if random.random() < p:\n        scale = 1 + np.random.uniform(-1, 1, size=(3, 1, 1, 1)) * np.array(scale)\n        return data_numpy * scale\n    else:\n        return data_numpy.copy()\n\n\n''' AimCLR '''\ntransform_order = {'ntu': [0, 1, 2, 3, 8, 9, 10, 11, 4, 5, 6, 7, 16, 17, 18, 19, 12, 13, 14, 15, 20, 23, 24, 21, 22]}\n\n\ndef subtract(data_numpy, p=0.5):\n    joint = random.randint(0, 24)\n\n    C, T, V, M = data_numpy.shape\n    if random.random() < p:\n        data_numpy_new = np.zeros((C, T, V, M))\n        for i in range(V):\n            data_numpy_new[:, :, i, :] = data_numpy[:, :, i, :] - data_numpy[:, :, joint, :]\n        return data_numpy_new\n    else:\n        return data_numpy.copy()\n\n\ndef temporal_flip(data_numpy, index_t, p=0.5):\n    C, T, V, M = data_numpy.shape\n    if random.random() < p:\n        time_range_order = [i for i in range(T)]\n        time_range_reverse = list(reversed(time_range_order))\n        return data_numpy[:, time_range_reverse, :, :], -index_t\n    else:\n        return data_numpy.copy(), index_t.copy()\n\n\ndef spatial_flip(data_numpy, p=0.5):\n    if random.random() < p:\n        index = transform_order['ntu']\n        return data_numpy[:, :, index, :]\n    else:\n        return data_numpy.copy()\n\n\ndef rotate(data_numpy, axis=None, angle=None, p=0.5):\n    if axis != None:\n        axis_next = axis\n    else:\n        axis_next = random.randint(0, 2)\n\n    if angle != None:\n        angle_next = random.uniform(-angle, angle)\n    else:\n        angle_next = random.uniform(-30, 30)\n\n    if random.random() < p:\n        temp = data_numpy.copy()\n        angle = math.radians(angle_next)\n        # x\n        if axis_next == 0:\n            R = np.array([[1, 0, 0],\n                          [0, math.cos(angle), math.sin(angle)],\n                          [0, -math.sin(angle), math.cos(angle)]])\n        # y\n        if axis_next == 1:\n            R = np.array([[math.cos(angle), 0, -math.sin(angle)],\n                          [0, 1, 0],\n                          [math.sin(angle), 0, math.cos(angle)]])\n        # z\n        if axis_next == 2:\n            R = np.array([[math.cos(angle), math.sin(angle), 0],\n                          [-math.sin(angle), math.cos(angle), 0],\n                          [0, 0, 1]])\n        R = R.transpose()\n        temp = np.dot(temp.transpose([1, 2, 3, 0]), R)\n        temp = temp.transpose(3, 0, 1, 2)\n        return temp\n    else:\n        return data_numpy.copy()\n\n\ndef shear(data_numpy, s1=None, s2=None, p=0.5):\n    if random.random() < p:\n        temp = data_numpy.copy()\n        if s1 != None:\n            s1_list = s1\n        else:\n            s1_list = [random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5)]\n        if s2 != None:\n            s2_list = s2\n        else:\n            s2_list = [random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5)]\n\n        R = np.array([[1, s1_list[0], s2_list[0]],\n                      [s1_list[1], 1, s2_list[1]],\n                      [s1_list[2], s2_list[2], 1]])\n        R = R.transpose()\n        temp = np.dot(temp.transpose([1, 2, 3, 0]), R)\n        temp = temp.transpose(3, 0, 1, 2)\n        return temp\n    else:\n        return data_numpy.copy()\n\n\ndef drop_axis(data_numpy, axis=None, p=0.5):\n    if axis != None:\n        axis_next = axis\n    else:\n        axis_next = random.randint(0, 2)\n\n    if random.random() < p:\n        temp = data_numpy.copy()\n        C, T, V, M = data_numpy.shape\n        x_new = np.zeros((T, V, M))\n        temp[axis_next] = x_new\n        return temp\n    else:\n        return data_numpy.copy()\n\n\ndef drop_joint(data_numpy, joint_list=None, time_range=None, p=0.5):\n    if random.random() < p:\n        temp = data_numpy.copy()\n        C, T, V, M = data_numpy.shape\n\n        if joint_list != None:\n            all_joints = [i for i in range(V)]\n            joint_list_ = random.sample(all_joints, joint_list)\n            joint_list_ = sorted(joint_list_)\n        else:\n            random_int = random.randint(5, 15)\n            all_joints = [i for i in range(V)]\n            joint_list_ = random.sample(all_joints, random_int)\n            joint_list_ = sorted(joint_list_)\n\n        if time_range != None:\n            all_frames = [i for i in range(T)]\n            time_range_ = random.sample(all_frames, time_range)\n            time_range_ = sorted(time_range_)\n        else:\n            random_int = random.randint(16, 32)\n            all_frames = [i for i in range(T)]\n            time_range_ = random.sample(all_frames, random_int)\n            time_range_ = sorted(time_range_)\n\n        x_new = np.zeros((C, len(time_range_), len(joint_list_), M))\n        temp2 = temp[:, time_range_, :, :].copy()\n        temp2[:, :, joint_list_, :] = x_new\n        temp[:, time_range_, :, :] = temp2\n        return temp\n    else:\n        return data_numpy.copy()\n\n\ndef gaussian_noise(data_numpy, mean=0, std=0.05, p=0.5):\n    if random.random() < p:\n        temp = data_numpy.copy()\n        C, T, V, M = data_numpy.shape\n        noise = np.random.normal(mean, std, size=(C, T, V, M))\n        return temp + noise\n    else:\n        return data_numpy.copy()\n\n\nclass GaussianBlurConv(nn.Module):\n    def __init__(self, channels=3, kernel=15, sigma=[0.1, 2], p=0.5):\n        super(GaussianBlurConv, self).__init__()\n        self.channels = channels\n        self.kernel = kernel\n        self.min_max_sigma = sigma\n        radius = int(kernel / 2)\n        self.kernel_index = np.arange(-radius, radius + 1)\n        self.p = p\n\n    def __call__(self, x):\n        sigma = random.uniform(self.min_max_sigma[0], self.min_max_sigma[1])\n        blur_flter = np.exp(-np.power(self.kernel_index, 2.0) / (2.0 * np.power(sigma, 2.0)))\n        kernel = torch.from_numpy(blur_flter).unsqueeze(0).unsqueeze(0)\n        # kernel =  kernel.float()\n        kernel = kernel.double()\n        kernel = kernel.repeat(self.channels, 1, 1, 1)\n        self.weight = nn.Parameter(data=kernel, requires_grad=False)\n\n        prob = np.random.random_sample()\n        x = torch.from_numpy(x).double()\n        if prob < self.p:\n            x = x.permute(3, 0, 2, 1)\n            x = F.conv2d(x, self.weight, padding=(0, int((self.kernel - 1) / 2)), groups=self.channels)\n            x = x.permute(1, -1, -2, 0)\n\n        return x.numpy()\n\n\ndef gaussian_filter(data_numpy, kernel=15, sig_list=[0.1, 2], p=0.5):\n    g = GaussianBlurConv(3, kernel, sig_list, p)\n    return g(data_numpy)\n\n\n''' Skeleton AdaIN '''\ndef skeleton_adain_bone_length(input, ref): # C T V M\n    eps = 1e-5\n    center = 1\n    ref_c = ref[:, :, center, :]\n\n    # joint to bone (joint2bone)\n    j2b = joint2bone()\n    bone_i = j2b(input) # C T V M\n    bone_r = j2b(ref)\n\n    bone_length_i = np.linalg.norm(bone_i, axis=0) # T V M\n    bone_length_r = np.linalg.norm(bone_r, axis=0)\n\n    bone_length_scale = (bone_length_r + eps) / (bone_length_i + eps) # T V M\n    bone_length_scale = np.expand_dims(bone_length_scale, axis=0) # 1 T V M\n\n    bone_i = bone_i * bone_length_scale\n\n    # bone to joint (bone2joint)\n    b2j = bone2joint()\n    joint = b2j(bone_i, ref_c)\n    return joint\n\n\nclass joint2bone(nn.Module):\n    def __init__(self):\n        super(joint2bone, self).__init__()\n        self.pairs = [(0, 1), (1, 1), (2, 20), (3, 2), (4, 20), (5, 4), (6, 5), (7, 6), (8, 20), (9, 8),\n                      (10, 9), (11, 10), (12, 0), (13, 12), (14, 13), (15, 14), (16, 0), (17, 16), (18, 17),\n                      (19, 18), (20, 1), (21, 7), (22, 7), (23, 11), (24, 11)]\n\n    def __call__(self, joint):\n        bone = np.zeros_like(joint)\n        for v1, v2 in self.pairs:\n            bone[:, :, v1, :] = joint[:, :, v1, :] - joint[:, :, v2, :]\n        return bone\n\n\nclass bone2joint(nn.Module):\n    def __init__(self):\n        super(bone2joint, self).__init__()\n        self.center = 1\n        self.pairs_1 = [(0, 1), (20, 1)]\n        self.pairs_2 = [(2, 20), (4, 20), (8, 20), (12, 0), (16, 0)]\n        self.pairs_3 = [(3, 2), (5, 4), (9, 8), (13, 12), (17, 16)]\n        self.pairs_4 = [(6, 5), (10, 9), (14, 13), (18, 17)]\n        self.pairs_5 = [(7, 6), (11, 10), (15, 14), (19, 18)]\n        self.pairs_6 = [(21, 7), (22, 7), (23, 11), (24, 11)]\n\n    def __call__(self, bone, center):\n        joint = np.zeros_like(bone)\n        joint[:, :, self.center, :] = center\n        for v1, v2 in self.pairs_1:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        for v1, v2 in self.pairs_2:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        for v1, v2 in self.pairs_3:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        for v1, v2 in self.pairs_4:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        for v1, v2 in self.pairs_5:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        for v1, v2 in self.pairs_6:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        return joint\n\ndef to_motion(input): # C T V M\n    C, T, V, M = input.shape\n    motion = np.zeros_like(input)\n    motion[:, :T - 1] = np.diff(input, axis=1)\n    return motion")

if "." not in sys.path:
    sys.path.insert(0, ".")

# 2. Định nghĩa hàm augmentation trong notebook scope
import random
import numpy as np
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


def valid_crop_resize(data_numpy, valid_frame_num, p_interval, window, thres):
    # input: C,T,V,M
    if isinstance(p_interval, (int, float)):
        p_interval = [p_interval]
    C, T, V, M = data_numpy.shape
    begin = 0
    end = valid_frame_num if valid_frame_num > 0 else T
    valid_size = max(1, end - begin)

    # crop
    if len(p_interval) == 1:
        p = p_interval[0]
        bias = int((1 - p) * valid_size / 2)
        c_b = min(begin + bias, T - 1)
        c_e = max(c_b + 1, min(end - bias, T))
        data = data_numpy[:, c_b:c_e, :, :]  # center_crop
        cropped_length = data.shape[1]
    else:
        p = np.random.rand(1) * (p_interval[1] - p_interval[0]) + p_interval[0]
        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres), valid_size)  # constraint cropped_length lower bound as thres
        max_bias = max(0, valid_size - cropped_length)
        bias = np.random.randint(0, max_bias + 1) if max_bias > 0 else 0
        c_b = begin + bias
        c_e = min(begin + bias + cropped_length, T)
        data = data_numpy[:, c_b:c_e, :, :]
        if data.shape[1] == 0:
            print(cropped_length, bias, valid_size)

    # resize
    data = torch.tensor(data, dtype=torch.float)  # C, crop_t, V, M
    data = data.permute(2, 3, 0, 1).contiguous().view(V * M, C, cropped_length)  # V*M, C, crop_t
    data = F.interpolate(data, size=window, mode='linear', align_corners=False)  # V*M, C, T
    data = data.contiguous().view(V, M, C, window).permute(2, 3, 0, 1).contiguous().numpy()
    index_t = torch.arange(start=c_b, end=c_e, dtype=torch.float)
    index_t = F.interpolate(index_t[None, None, :], size=window, mode='linear', align_corners=False).squeeze()
    index_t = 2 * index_t / valid_size - 1
    return data, index_t.numpy()


def valid_crop_uniform(data_numpy, valid_frame_num, p_interval, window, thres):
    # input: C,T,V,M
    if isinstance(p_interval, (int, float)):
        p_interval = [p_interval]
    C, T, V, M = data_numpy.shape
    begin = 0
    end = valid_frame_num if valid_frame_num > 0 else T
    valid_size = max(1, end - begin)

    # crop
    if len(p_interval) == 1:
        p = p_interval[0]
        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres), valid_size)
        bias = int((1 - p) * valid_size / 2)
        bias = min(bias, max(0, valid_size - cropped_length))

        if cropped_length < window:
            inds = np.arange(cropped_length)
        else:
            bids = np.array(
                [i * cropped_length // window for i in range(window + 1)])
            bst = bids[:window]
            inds = bst

        inds = inds + bias
        inds = np.clip(inds, 0, T - 1)
        data = data_numpy[:, inds, :, :]

    else:
        p = np.random.rand(1) * (p_interval[1] - p_interval[0]) + p_interval[0]
        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres),
                                    valid_size)  # constraint cropped_length lower bound as 64
        max_bias = max(0, valid_size - cropped_length)
        bias = np.random.randint(0, max_bias + 1) if max_bias > 0 else 0

        if cropped_length < window:
            inds = np.arange(cropped_length)
        elif window <= cropped_length < 2 * window:
            basic = np.arange(window)
            inds = np.random.choice(window + 1, cropped_length - window, replace=False)
            offset = np.zeros(window + 1, dtype=np.int64)
            offset[inds] = 1
            offset = np.cumsum(offset)
            inds = basic + offset[:-1]
        else:
            bids = np.array([i * cropped_length // window for i in range(window + 1)])
            bsize = np.diff(bids)
            bst = bids[:window]
            offset = np.random.randint(bsize)
            inds = bst + offset

        inds = inds + bias
        inds = np.clip(inds, 0, T - 1)
        data = data_numpy[:, inds, :, :]
        if data.shape[1] == 0:
            print(cropped_length, bias, valid_size)

    # resize
    data = torch.tensor(data, dtype=torch.float)
    index_t = torch.tensor(inds, dtype=torch.float)
    data = data.permute(2, 3, 0, 1).contiguous().view(V * M, C, len(inds))  # V*M, C, crop_t

    if len(inds) != window:
        data = F.interpolate(data, size=window, mode='linear', align_corners=False)  # V*M, C, T
        index_t = F.interpolate(index_t[None, None, :], size=window, mode='linear', align_corners=False).squeeze()

    data = data.contiguous().view(V, M, C, window).permute(2, 3, 0, 1).contiguous().numpy()
    index_t = 2 * index_t / valid_size - 1
    return data, index_t.numpy()


def scale(data_numpy, scale=0.2, p=0.5):
    if random.random() < p:
        scale = 1 + np.random.uniform(-1, 1, size=(3, 1, 1, 1)) * np.array(scale)
        return data_numpy * scale
    else:
        return data_numpy.copy()


''' AimCLR '''
transform_order = {'ntu': [0, 1, 2, 3, 8, 9, 10, 11, 4, 5, 6, 7, 16, 17, 18, 19, 12, 13, 14, 15, 20, 23, 24, 21, 22]}


def subtract(data_numpy, p=0.5):
    joint = random.randint(0, 24)

    C, T, V, M = data_numpy.shape
    if random.random() < p:
        data_numpy_new = np.zeros((C, T, V, M))
        for i in range(V):
            data_numpy_new[:, :, i, :] = data_numpy[:, :, i, :] - data_numpy[:, :, joint, :]
        return data_numpy_new
    else:
        return data_numpy.copy()


def temporal_flip(data_numpy, index_t, p=0.5):
    C, T, V, M = data_numpy.shape
    if random.random() < p:
        time_range_order = [i for i in range(T)]
        time_range_reverse = list(reversed(time_range_order))
        return data_numpy[:, time_range_reverse, :, :], -index_t
    else:
        return data_numpy.copy(), index_t.copy()


def spatial_flip(data_numpy, p=0.5):
    if random.random() < p:
        index = transform_order['ntu']
        return data_numpy[:, :, index, :]
    else:
        return data_numpy.copy()


def rotate(data_numpy, axis=None, angle=None, p=0.5):
    if axis != None:
        axis_next = axis
    else:
        axis_next = random.randint(0, 2)

    if angle != None:
        angle_next = random.uniform(-angle, angle)
    else:
        angle_next = random.uniform(-30, 30)

    if random.random() < p:
        temp = data_numpy.copy()
        angle = math.radians(angle_next)
        # x
        if axis_next == 0:
            R = np.array([[1, 0, 0],
                          [0, math.cos(angle), math.sin(angle)],
                          [0, -math.sin(angle), math.cos(angle)]])
        # y
        if axis_next == 1:
            R = np.array([[math.cos(angle), 0, -math.sin(angle)],
                          [0, 1, 0],
                          [math.sin(angle), 0, math.cos(angle)]])
        # z
        if axis_next == 2:
            R = np.array([[math.cos(angle), math.sin(angle), 0],
                          [-math.sin(angle), math.cos(angle), 0],
                          [0, 0, 1]])
        R = R.transpose()
        temp = np.dot(temp.transpose([1, 2, 3, 0]), R)
        temp = temp.transpose(3, 0, 1, 2)
        return temp
    else:
        return data_numpy.copy()


def shear(data_numpy, s1=None, s2=None, p=0.5):
    if random.random() < p:
        temp = data_numpy.copy()
        if s1 != None:
            s1_list = s1
        else:
            s1_list = [random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5)]
        if s2 != None:
            s2_list = s2
        else:
            s2_list = [random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5)]

        R = np.array([[1, s1_list[0], s2_list[0]],
                      [s1_list[1], 1, s2_list[1]],
                      [s1_list[2], s2_list[2], 1]])
        R = R.transpose()
        temp = np.dot(temp.transpose([1, 2, 3, 0]), R)
        temp = temp.transpose(3, 0, 1, 2)
        return temp
    else:
        return data_numpy.copy()


def drop_axis(data_numpy, axis=None, p=0.5):
    if axis != None:
        axis_next = axis
    else:
        axis_next = random.randint(0, 2)

    if random.random() < p:
        temp = data_numpy.copy()
        C, T, V, M = data_numpy.shape
        x_new = np.zeros((T, V, M))
        temp[axis_next] = x_new
        return temp
    else:
        return data_numpy.copy()


def drop_joint(data_numpy, joint_list=None, time_range=None, p=0.5):
    if random.random() < p:
        temp = data_numpy.copy()
        C, T, V, M = data_numpy.shape

        if joint_list != None:
            all_joints = [i for i in range(V)]
            joint_list_ = random.sample(all_joints, joint_list)
            joint_list_ = sorted(joint_list_)
        else:
            random_int = random.randint(5, 15)
            all_joints = [i for i in range(V)]
            joint_list_ = random.sample(all_joints, random_int)
            joint_list_ = sorted(joint_list_)

        if time_range != None:
            all_frames = [i for i in range(T)]
            time_range_ = random.sample(all_frames, time_range)
            time_range_ = sorted(time_range_)
        else:
            random_int = random.randint(16, 32)
            all_frames = [i for i in range(T)]
            time_range_ = random.sample(all_frames, random_int)
            time_range_ = sorted(time_range_)

        x_new = np.zeros((C, len(time_range_), len(joint_list_), M))
        temp2 = temp[:, time_range_, :, :].copy()
        temp2[:, :, joint_list_, :] = x_new
        temp[:, time_range_, :, :] = temp2
        return temp
    else:
        return data_numpy.copy()


def gaussian_noise(data_numpy, mean=0, std=0.05, p=0.5):
    if random.random() < p:
        temp = data_numpy.copy()
        C, T, V, M = data_numpy.shape
        noise = np.random.normal(mean, std, size=(C, T, V, M))
        return temp + noise
    else:
        return data_numpy.copy()


class GaussianBlurConv(nn.Module):
    def __init__(self, channels=3, kernel=15, sigma=[0.1, 2], p=0.5):
        super(GaussianBlurConv, self).__init__()
        self.channels = channels
        self.kernel = kernel
        self.min_max_sigma = sigma
        radius = int(kernel / 2)
        self.kernel_index = np.arange(-radius, radius + 1)
        self.p = p

    def __call__(self, x):
        sigma = random.uniform(self.min_max_sigma[0], self.min_max_sigma[1])
        blur_flter = np.exp(-np.power(self.kernel_index, 2.0) / (2.0 * np.power(sigma, 2.0)))
        kernel = torch.from_numpy(blur_flter).unsqueeze(0).unsqueeze(0)
        # kernel =  kernel.float()
        kernel = kernel.double()
        kernel = kernel.repeat(self.channels, 1, 1, 1)
        self.weight = nn.Parameter(data=kernel, requires_grad=False)

        prob = np.random.random_sample()
        x = torch.from_numpy(x).double()
        if prob < self.p:
            x = x.permute(3, 0, 2, 1)
            x = F.conv2d(x, self.weight, padding=(0, int((self.kernel - 1) / 2)), groups=self.channels)
            x = x.permute(1, -1, -2, 0)

        return x.numpy()


def gaussian_filter(data_numpy, kernel=15, sig_list=[0.1, 2], p=0.5):
    g = GaussianBlurConv(3, kernel, sig_list, p)
    return g(data_numpy)


''' Skeleton AdaIN '''
def skeleton_adain_bone_length(input, ref): # C T V M
    eps = 1e-5
    center = 1
    ref_c = ref[:, :, center, :]

    # joint to bone (joint2bone)
    j2b = joint2bone()
    bone_i = j2b(input) # C T V M
    bone_r = j2b(ref)

    bone_length_i = np.linalg.norm(bone_i, axis=0) # T V M
    bone_length_r = np.linalg.norm(bone_r, axis=0)

    bone_length_scale = (bone_length_r + eps) / (bone_length_i + eps) # T V M
    bone_length_scale = np.expand_dims(bone_length_scale, axis=0) # 1 T V M

    bone_i = bone_i * bone_length_scale

    # bone to joint (bone2joint)
    b2j = bone2joint()
    joint = b2j(bone_i, ref_c)
    return joint


class joint2bone(nn.Module):
    def __init__(self):
        super(joint2bone, self).__init__()
        self.pairs = [(0, 1), (1, 1), (2, 20), (3, 2), (4, 20), (5, 4), (6, 5), (7, 6), (8, 20), (9, 8),
                      (10, 9), (11, 10), (12, 0), (13, 12), (14, 13), (15, 14), (16, 0), (17, 16), (18, 17),
                      (19, 18), (20, 1), (21, 7), (22, 7), (23, 11), (24, 11)]

    def __call__(self, joint):
        bone = np.zeros_like(joint)
        for v1, v2 in self.pairs:
            bone[:, :, v1, :] = joint[:, :, v1, :] - joint[:, :, v2, :]
        return bone


class bone2joint(nn.Module):
    def __init__(self):
        super(bone2joint, self).__init__()
        self.center = 1
        self.pairs_1 = [(0, 1), (20, 1)]
        self.pairs_2 = [(2, 20), (4, 20), (8, 20), (12, 0), (16, 0)]
        self.pairs_3 = [(3, 2), (5, 4), (9, 8), (13, 12), (17, 16)]
        self.pairs_4 = [(6, 5), (10, 9), (14, 13), (18, 17)]
        self.pairs_5 = [(7, 6), (11, 10), (15, 14), (19, 18)]
        self.pairs_6 = [(21, 7), (22, 7), (23, 11), (24, 11)]

    def __call__(self, bone, center):
        joint = np.zeros_like(bone)
        joint[:, :, self.center, :] = center
        for v1, v2 in self.pairs_1:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_2:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_3:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_4:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_5:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_6:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        return joint

def to_motion(input): # C T V M
    C, T, V, M = input.shape
    motion = np.zeros_like(input)
    motion[:, :T - 1] = np.diff(input, axis=1)
    return motion

# 3. Đăng ký module 'tools' vào sys.modules và globals để cả gọi trực tiếp lẫn qua tools.<func> đều hoạt động 100%
tools = types.ModuleType("tools")
for _k, _v in list(globals().items()):
    if not _k.startswith("_"):
        setattr(tools, _k, _v)
sys.modules["tools"] = tools
sys.modules["feeders.tools"] = tools

feeders_mod = types.ModuleType("feeders")
setattr(feeders_mod, "tools", tools)
sys.modules["feeders"] = feeders_mod
print("Data augmentation tools ready and feeders/tools.py written to disk!")



In [ ]:
# NTU RGB+D Feeder (giống hệt CLIR-SkateFormer, khong doi khi doi backbone)
import numpy as np
import random
import csv

from torch.utils.data import Dataset
import sys, types
if '.' not in sys.path:
    sys.path.insert(0, '.')
try:
    from feeders import tools
except Exception:
    if 'tools' not in globals() or tools is None:
        tools = sys.modules.get('tools', sys.modules.get('feeders.tools', None))



class Feeder(Dataset):
    def __init__(self, data_path, label_path=None, p_interval=1, split='train', data_type='j',
                 aug_method='z', intra_p=0.5, inter_p=0.0, window_size=-1,
                 debug=False, thres=64, uniform=False, partition=False,
                 concepts_csv=None, remove_null_concept=True):

        self.debug = debug
        self.data_path = data_path
        self.label_path = label_path
        self.split = split
        self.data_type = data_type
        self.aug_method = aug_method
        self.intra_p = intra_p
        self.inter_p = inter_p
        self.window_size = window_size
        self.p_interval = p_interval
        self.thres = thres
        self.uniform = uniform
        self.partition = partition
        self.concepts_csv = concepts_csv
        self.remove_null_concept = remove_null_concept
        self.load_data()
        self.concept_matrix = None
        self.concept_names = None
        if self.concepts_csv:
            self.load_concepts()
        if partition:
            self.right_arm = np.array([7, 8, 22, 23]) - 1
            self.left_arm = np.array([11, 12, 24, 25]) - 1
            self.right_leg = np.array([13, 14, 15, 16]) - 1
            self.left_leg = np.array([17, 18, 19, 20]) - 1
            self.h_torso = np.array([5, 9, 6, 10]) - 1
            self.w_torso = np.array([2, 3, 1, 4]) - 1
            self.new_idx = np.concatenate((self.right_arm, self.left_arm, self.right_leg, self.left_leg, self.h_torso, self.w_torso), axis=-1)
            # except for joint no.21

    def load_concepts(self):
        """Load REASON concept targets without changing the legacy feeder API."""
        with open(self.concepts_csv, 'r', newline='', encoding='utf-8-sig') as handle:
            rows = list(csv.DictReader(handle))

        if not rows or 'action_class' not in rows[0]:
            raise ValueError(f'Invalid concepts CSV: {self.concepts_csv}')

        names = [name for name in rows[0].keys() if name != 'action_class']
        # Nếu CSV có 74 concepts (chưa loại null concepts), lọc bỏ 2 null concepts
        # Nếu CSV đã là 72 concepts chuẩn, giữ nguyên 72 concepts
        if self.remove_null_concept and len(names) == 74:
            names = [name for i, name in enumerate(names) if i not in (50, 65)]
        elif self.remove_null_concept and len(names) > 72:
            names = [name for name in names if name not in ('foot_tap', 'temporal_sequence_seq_cascading')]

        values = []
        for row in rows:
            values.append([float(row[name]) for name in names])
        self.concept_names = names
        self.concept_matrix = np.asarray(values, dtype=np.float32)
        if self.concept_matrix.shape[1] != 72:
            raise ValueError(
                f'Expected 72 concepts after filtering, got {self.concept_matrix.shape[1]}'
            )

    def load_data(self):
        # data: N C V T M
        npz_data = np.load(self.data_path)
        if self.split == 'train':
            self.data = npz_data['x_train']
            self.label = np.where(npz_data['y_train'] > 0)[1]
            self.sample_name = ['train_' + str(i) for i in range(len(self.data))]
        elif self.split == 'test':
            self.data = npz_data['x_test']
            self.label = np.where(npz_data['y_test'] > 0)[1]
            self.sample_name = ['test_' + str(i) for i in range(len(self.data))]
        else:
            raise NotImplementedError('data split only supports train/test')
        N, T, _ = self.data.shape
        self.data = self.data.reshape((N, T, 2, 25, 3)).transpose(0, 4, 1, 3, 2)

    def __len__(self):
        return len(self.label)

    def __iter__(self):
        return self

    def __getitem__(self, index):
        global tools
        if 'tools' not in globals() or tools is None:
            import sys
            try:
                from feeders import tools
            except Exception:
                tools = sys.modules.get('tools', sys.modules.get('feeders.tools', None))

        data_numpy = self.data[index]
        label = self.label[index]
        data_numpy = np.array(data_numpy)
        valid_frame_num = np.sum(data_numpy.sum(0).sum(-1).sum(-1) != 0)
        num_people = np.sum(data_numpy.sum(0).sum(0).sum(0) != 0)

        if self.uniform:
            data_numpy, index_t = tools.valid_crop_uniform(data_numpy, valid_frame_num, self.p_interval,
                                                           self.window_size, self.thres)
        else:
            data_numpy, index_t = tools.valid_crop_resize(data_numpy, valid_frame_num, self.p_interval,
                                                          self.window_size, self.thres)

        if self.split == 'train':
            # intra-instance augmentation
            p = np.random.rand(1)
            if p < self.intra_p:

                if 'a' in self.aug_method:
                    if np.random.rand(1) < 0.5:
                        data_numpy = data_numpy[:, :, :, np.array([1, 0])]
                if 'b' in self.aug_method:
                    if num_people == 2:
                        if np.random.rand(1) < 0.5:
                            axis_next = np.random.randint(0, 1)
                            temp = data_numpy.copy()
                            C, T, V, M = data_numpy.shape
                            x_new = np.zeros((C, T, V))
                            temp[:, :, :, axis_next] = x_new
                            data_numpy = temp

                if '1' in self.aug_method:
                    data_numpy = tools.shear(data_numpy, p=0.5)
                if '2' in self.aug_method:
                    data_numpy = tools.rotate(data_numpy, p=0.5)
                if '3' in self.aug_method:
                    data_numpy = tools.scale(data_numpy, p=0.5)
                if '4' in self.aug_method:
                    data_numpy = tools.spatial_flip(data_numpy, p=0.5)
                if '5' in self.aug_method:
                    data_numpy, index_t = tools.temporal_flip(data_numpy, index_t, p=0.5)
                if '6' in self.aug_method:
                    data_numpy = tools.gaussian_noise(data_numpy, p=0.5)
                if '7' in self.aug_method:
                    data_numpy = tools.gaussian_filter(data_numpy, p=0.5)
                if '8' in self.aug_method:
                    data_numpy = tools.drop_axis(data_numpy, p=0.5)
                if '9' in self.aug_method:
                    data_numpy = tools.drop_joint(data_numpy, p=0.5)

            # inter-instance augmentation
            elif (p < (self.intra_p + self.inter_p)) & (p >= self.intra_p):
                adain_idx = random.choice(np.where(self.label == label)[0])
                data_adain = self.data[adain_idx]
                data_adain = np.array(data_adain)
                f_num = np.sum(data_adain.sum(0).sum(-1).sum(-1) != 0)
                t_idx = np.round((index_t + 1) * f_num / 2).astype(np.int64)
                data_adain = data_adain[:, t_idx]
                data_numpy = tools.skeleton_adain_bone_length(data_numpy, data_adain)

            else:
                data_numpy = data_numpy.copy()

        # modality
        if self.data_type == 'b':
            j2b = tools.joint2bone()
            data_numpy = j2b(data_numpy)
        elif self.data_type == 'jm':
            data_numpy = tools.to_motion(data_numpy)
        elif self.data_type == 'bm':
            j2b = tools.joint2bone()
            data_numpy = j2b(data_numpy)
            data_numpy = tools.to_motion(data_numpy)
        else:
            data_numpy = data_numpy.copy()

        if self.partition:
            data_numpy = data_numpy[:, :, self.new_idx]

        if self.concept_matrix is not None:
            if label >= len(self.concept_matrix):
                raise IndexError(f'Concept CSV has no row for action label {label}')
            return data_numpy, index_t, label, index, self.concept_matrix[label]
        return data_numpy, index_t, label, index

    def top_k(self, score, top_k):
        rank = score.argsort()
        hit_top_k = [l in rank[i, -top_k:] for i, l in enumerate(self.label)]
        return sum(hit_top_k) * 1.0 / len(hit_top_k)


def import_class(name):
    components = name.split('.')
    mod = __import__(components[0])
    for comp in components[1:]:
        mod = getattr(mod, comp)
    return mod




## 🧠 Block 3: HyperGCN Perception Backbone
Backbone NTU-25 chuẩn — `extract_feature_map()` trả về `[N, D, T', V=25]` (V không bị downsample qua backbone), dùng cho G1 ở Block 6.


In [ ]:
# HyperGCN Backbone Architecture
"""
Hyper-GCN Backbone Module (Complete Architecture)
=================================================
Hypergraph Convolutional Network with Multi-Scale Temporal Convolutions
and Virtual Joint Ensemble for Skeleton-Based Action Recognition.

Nguồn: kế thừa nguyên trạng từ `GRAFT-SAR/model/backbones/hypergcn.py`
(bản thân là bản làm sạch của `d:\\Thesis\\Hyper-GCN\\model\\hypergcn_large.py`).
Chỉ thêm `extract_feature_map()` để lộ ra feature map [N, D, T', V] trước khi
pool toàn cục — cần cho BodyPartGroundedConceptHeads (G1) của GLIR-HyperGCN,
vì V ở đây ánh xạ 1:1 với 25 khớp NTU chuẩn (virtual hyper-joints đã bị cắt bỏ
trong HYPERGC.forward trước khi trả về).
"""

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


def conv_branch_init(conv, branches):
    weight = conv.weight
    n = weight.size(0)
    k1 = weight.size(1)
    k2 = weight.size(2)
    nn.init.normal_(weight, 0, math.sqrt(2.0 / (n * k1 * k2 * branches)))
    if conv.bias is not None:
        nn.init.constant_(conv.bias, 0)


def conv_init(conv):
    if conv.weight is not None:
        nn.init.kaiming_normal_(conv.weight, mode="fan_out")
    if conv.bias is not None:
        nn.init.constant_(conv.bias, 0)


def bn_init(bn, scale):
    nn.init.constant_(bn.weight, scale)
    nn.init.constant_(bn.bias, 0)


def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        if hasattr(m, "weight"):
            nn.init.kaiming_normal_(m.weight, mode="fan_out")
        if hasattr(m, "bias") and m.bias is not None and isinstance(m.bias, torch.Tensor):
            nn.init.constant_(m.bias, 0)
    elif classname.find("BatchNorm") != -1:
        if hasattr(m, "weight") and m.weight is not None:
            m.weight.data.normal_(1.0, 0.02)
        if hasattr(m, "bias") and m.bias is not None:
            m.bias.data.fill_(0)


# =========================================================================
# Graph Adjacency Generation for HyperGCN
# =========================================================================

def edge2mat(link, num_node):
    A = np.zeros((num_node, num_node))
    for i, j in link:
        A[j, i] = 1
    return A


def get_virtual_graph_ensemble(num_node, self_link, inward, outward, virtual, nums=8):
    I = np.pad(edge2mat(self_link, num_node), ((0, virtual), (0, virtual)))
    In = np.pad(edge2mat(inward, num_node), ((0, virtual), (0, virtual)))
    Out = np.pad(edge2mat(outward, num_node), ((0, virtual), (0, virtual)))
    for i in range(virtual):
        I[num_node + i, num_node + i] = 1
        In[:num_node, num_node + i] = 1
        Out[num_node + i, :num_node] = 1
    A = I + In + Out
    A = np.repeat(A[np.newaxis, :], nums, axis=0)
    return A


class HyperGraph:
    def __init__(self, num_node=25, hyper_joints=3, labeling_mode="virtual_ensemble"):
        self.num_node = num_node
        self.hyper_joints = hyper_joints
        self.self_link = [(i, i) for i in range(num_node)]
        inward_ori_index = [
            (1, 2), (2, 21), (3, 21), (4, 3), (5, 21), (6, 5), (7, 6),
            (8, 7), (9, 21), (10, 9), (11, 10), (12, 11), (13, 1),
            (14, 13), (15, 14), (16, 15), (17, 1), (18, 17), (19, 18),
            (20, 19), (22, 8), (23, 8), (24, 12), (25, 12)
        ]
        self.inward = [(i - 1, j - 1) for (i, j) in inward_ori_index]
        self.outward = [(j, i) for (i, j) in self.inward]
        self.neighbor = self.inward + self.outward
        self.A = get_virtual_graph_ensemble(
            self.num_node, self.self_link, self.inward, self.outward, self.hyper_joints, nums=8
        )


# =========================================================================
# Hypergraph Convolution Unit
# =========================================================================

class HYPERGC(nn.Module):
    def __init__(self, in_channels, out_channels, vertex_nums=25, virtual_num=3, A=None, hyper=True, num_subset=8, rel_reduction=4):
        super(HYPERGC, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.vertex_nums = vertex_nums
        self.virtual_num = virtual_num
        self.rel_reduction = rel_reduction
        self.num_subset = num_subset
        self.hyper = hyper
        mid_in_channels = in_channels // num_subset
        mid_out_channels = out_channels // num_subset
        self.mid_in_channels = mid_in_channels
        self.mid_out_channels = mid_out_channels

        if self.hyper:
            self.hidden_channels = mid_in_channels // rel_reduction
            self.to_V = nn.Conv1d(in_channels, num_subset * self.hidden_channels, kernel_size=1, groups=num_subset)
            self.to_W = nn.Sequential(
                nn.Conv1d(in_channels, num_subset * self.hidden_channels, kernel_size=1, groups=num_subset),
                nn.LeakyReLU(),
                nn.Conv1d(num_subset * self.hidden_channels, num_subset, kernel_size=1),
                nn.Tanh()
            )
            self.hyper_joint = nn.Parameter(torch.zeros(self.virtual_num, in_channels))
            self.alpha = nn.Parameter(torch.ones(1))
            self.softmax = nn.Softmax(dim=-1)

        self.conv_d = nn.Conv2d(in_channels, out_channels, kernel_size=1, groups=num_subset)
        if A is not None:
            self.PA = nn.Parameter(torch.from_numpy(A.astype(np.float32)), requires_grad=False)
            self.edge_importance = nn.Parameter(torch.ones(A.shape))
        else:
            self.PA = None
            self.edge_importance = None

        if in_channels != out_channels:
            self.down = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.down = lambda x: x
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                conv_init(m)
            elif isinstance(m, nn.BatchNorm2d):
                bn_init(m, 1)
        bn_init(self.bn, 1e-6)
        if self.hyper:
            conv_init(self.to_V)
            conv_init(self.to_W[0])
            conv_init(self.to_W[2])
        conv_init(self.conv_d)

    def hyper_norm(self, H, W):
        # Cast to float32 to avoid FP16 overflow under AMP
        H = H.float()
        W = W.float()
        w = torch.diag_embed(W)
        norm_w = torch.norm(H, 1, dim=2, keepdim=True).clamp_min(1e-8)
        w_ = w / norm_w

        H_w = H @ w
        norm_v = torch.norm(H_w, 1, dim=3, keepdim=True).clamp_min(1e-8)
        h_ = H_w / norm_v
        A = h_ @ w_ @ H.transpose(3, 2)
        # Guard against any residual NaN/Inf
        A = torch.nan_to_num(A, nan=0.0, posinf=1.0, neginf=-1.0)
        return A

    def a_norm(self, A):
        d_r = torch.norm(A, 1, dim=2, keepdim=True) + 1e-8
        return A / d_r

    def forward(self, x):
        N, C, T, V = x.size()

        h_x = self.hyper_joint
        h_x = (h_x.T).unsqueeze(1)
        x = torch.cat([x, h_x.repeat(N, 1, T, 1)], dim=-1)
        V += self.virtual_num
        A = self.PA.to(x.device)
        A = self.edge_importance * A
        A = self.a_norm(A)

        if self.hyper:
            t_x = x.mean(2)
            v_x = self.to_V(t_x)

            dis_v_x = v_x.view(N, self.num_subset, self.hidden_channels, V)
            dis_v_x = dis_v_x.permute(0, 1, 3, 2).contiguous().float()
            # Stable pairwise distance: avoids gradient explosion at d=0 from torch.cdist
            diff = dis_v_x.unsqueeze(3) - dis_v_x.unsqueeze(2)          # (N,K,V,V,hidden)
            distance_x = torch.sqrt((diff * diff).sum(-1) + 1e-6)        # (N,K,V,V)
            H = torch.zeros_like(distance_x)

            topk_v, topk_indices = torch.topk(distance_x, min(9, V), largest=False)
            topk_v = self.softmax(-topk_v)
            H = torch.scatter(H, 3, topk_indices, topk_v)

            W = self.to_W(t_x)
            H = self.hyper_norm(H, W)          # returns float32
            alpha = self.relu(self.alpha).float()
            A = A.float() + alpha * H
            A = A.to(x.dtype)                  # restore original dtype

        d_x = self.conv_d(x)
        d_x = d_x.view(N, self.num_subset, self.mid_out_channels, T, V)
        y = torch.einsum("nkuv,nkctv->nkctu", A, d_x).contiguous()
        y = y.view(N, self.out_channels, T, V)

        x = x[..., :self.vertex_nums]
        y = y[..., :self.vertex_nums]

        y = self.bn(y)
        y += self.down(x)
        y = self.relu(y)
        return y, self.hyper_joint


# =========================================================================
# Multi-Scale Temporal Convolution
# =========================================================================

class TemporalConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, dilation=1):
        super(TemporalConv, self).__init__()
        pad = (kernel_size + (kernel_size - 1) * (dilation - 1) - 1) // 2
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(kernel_size, 1),
            padding=(pad, 0),
            stride=(stride, 1),
            dilation=(dilation, 1),
            padding_mode="replicate",
        )
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        return self.bn(self.conv(x))


class MultiScale_TemporalConv(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
        stride=1,
        dilations=[1, 2, 3, 4],
        residual=True,
        residual_kernel_size=1,
    ):
        super().__init__()
        assert out_channels % (len(dilations) + 2) == 0, "# out channels should be multiples of # branches"

        self.num_branches = len(dilations) + 2
        branch_channels = out_channels // self.num_branches

        if not isinstance(kernel_size, list):
            kernel_size = [kernel_size] * len(dilations)

        self.branches = nn.ModuleList()
        for ks, dilation in zip(kernel_size, dilations):
            self.branches.append(
                nn.Sequential(
                    nn.Conv2d(in_channels, branch_channels, kernel_size=1, padding=0),
                    nn.BatchNorm2d(branch_channels),
                    nn.ReLU(inplace=True),
                    TemporalConv(branch_channels, branch_channels, kernel_size=ks, stride=stride, dilation=dilation),
                )
            )

        self.branches.append(
            nn.Sequential(
                nn.Conv2d(in_channels, branch_channels, kernel_size=1, padding=0),
                nn.BatchNorm2d(branch_channels),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=(3, 1), stride=(stride, 1), padding=(1, 0)),
                nn.BatchNorm2d(branch_channels),
            )
        )

        self.branches.append(
            nn.Sequential(
                nn.Conv2d(in_channels, branch_channels, kernel_size=1, padding=0, stride=(stride, 1)),
                nn.BatchNorm2d(branch_channels),
            )
        )

        if not residual:
            self.residual = lambda x: 0
        elif (in_channels == out_channels) and (stride == 1):
            self.residual = lambda x: x
        else:
            self.residual = TemporalConv(in_channels, out_channels, kernel_size=residual_kernel_size, stride=stride)

        self.apply(weights_init)

    def forward(self, x):
        res = self.residual(x)
        branch_outs = [tempconv(x) for tempconv in self.branches]
        out = torch.cat(branch_outs, dim=1)
        out += res
        return out


class TCN_GCN_unit(nn.Module):
    def __init__(self, in_channels, out_channels, num_point, hyper_joints, A, stride=1, residual=True, kernel_size=5, dilations=[1, 2], hyper=True):
        super(TCN_GCN_unit, self).__init__()
        self.gcn1 = HYPERGC(in_channels, out_channels, num_point, hyper_joints, A, hyper=hyper)
        self.tcn1 = MultiScale_TemporalConv(
            out_channels, out_channels, kernel_size=kernel_size, stride=stride, dilations=dilations, residual=False
        )
        self.relu = nn.ReLU(inplace=True)
        if not residual:
            self.residual = lambda x: 0
        elif (in_channels == out_channels) and (stride == 1):
            self.residual = lambda x: x
        else:
            self.residual = TemporalConv(in_channels, out_channels, kernel_size=1, stride=stride)

    def forward(self, x):
        y, h_x = self.gcn1(x)
        y = self.relu(self.tcn1(y) + self.residual(x))
        return y, h_x


# =========================================================================
# HyperGCN Full Model
# =========================================================================

class HyperGCNModel(nn.Module):
    """Full Hyper-GCN Model with 128 embedding channels and 9 spatial-temporal blocks."""

    def __init__(
        self,
        num_class=60,
        num_point=25,
        num_person=2,
        in_channels=3,
        hyper_joints=3,
        embedding_channels=128,
        drop_out=0.0,
        **kwargs,
    ):
        super(HyperGCNModel, self).__init__()
        self.num_class = num_class
        self.num_point = num_point
        self.embedding_channels = embedding_channels
        self.hyper_joints = hyper_joints

        self.graph = HyperGraph(num_node=num_point, hyper_joints=hyper_joints)
        A = self.graph.A

        self.data_bn = nn.BatchNorm1d(num_person * self.embedding_channels * num_point)
        self.to_joint_embedding = nn.Linear(in_channels, self.embedding_channels)
        self.pos_embedding = nn.Parameter(torch.randn(1, self.num_point, self.embedding_channels))
        self.tanh = nn.Tanh()

        self.l1 = TCN_GCN_unit(self.embedding_channels, self.embedding_channels, num_point, hyper_joints, A)
        self.l2 = TCN_GCN_unit(self.embedding_channels, self.embedding_channels, num_point, hyper_joints, A)
        self.l3 = TCN_GCN_unit(self.embedding_channels, self.embedding_channels, num_point, hyper_joints, A)
        self.l4 = TCN_GCN_unit(self.embedding_channels, self.embedding_channels * 2, num_point, hyper_joints, A, stride=2)
        self.l5 = TCN_GCN_unit(self.embedding_channels * 2, self.embedding_channels * 2, num_point, hyper_joints, A)
        self.l6 = TCN_GCN_unit(self.embedding_channels * 2, self.embedding_channels * 2, num_point, hyper_joints, A)
        self.l7 = TCN_GCN_unit(self.embedding_channels * 2, self.embedding_channels * 4, num_point, hyper_joints, A, stride=2)
        self.l8 = TCN_GCN_unit(self.embedding_channels * 4, self.embedding_channels * 4, num_point, hyper_joints, A)
        self.l9 = TCN_GCN_unit(self.embedding_channels * 4, self.embedding_channels * 4, num_point, hyper_joints, A)

        self.feat_dim = self.embedding_channels * 4  # 512
        self.fc = nn.Linear(self.feat_dim, self.num_class)

        nn.init.normal_(self.fc.weight, 0, math.sqrt(2.0 / num_class))
        bn_init(self.data_bn, 1)
        self.drop_out = nn.Dropout(drop_out) if drop_out > 0 else nn.Identity()

    def _backbone_layers(self, x, N, M):
        """Chạy 9 khối TCN_GCN, trả về feature map [N, c_new, T', V] (đã average qua M người,
        CHƯA pool qua T'/V) — V vẫn ánh xạ 1:1 với num_point khớp gốc."""
        x, _ = self.l1(x)
        x1 = x
        x, _ = self.l2(x)
        x, _ = self.l3(x + x1)

        x, _ = self.l4(x)
        x4 = x
        x, _ = self.l5(x)
        x, _ = self.l6(x + x4)

        x, _ = self.l7(x)
        x7 = x
        x, _ = self.l8(x)
        x, _ = self.l9(x + x7)

        c_new, T_, V_ = x.size(1), x.size(2), x.size(3)
        x = x.view(N, M, c_new, T_, V_)
        feature_map = x.mean(dim=1)  # average qua M người -> [N, c_new, T', V]
        return feature_map

    def _embed_input(self, x):
        N, C, T, V, M = x.size()
        x = x.permute(0, 4, 2, 3, 1).contiguous()
        x = self.to_joint_embedding(x)
        x = x + self.pos_embedding[:, :V]
        x = self.tanh(x).permute(0, 1, 3, 4, 2).contiguous()

        x = x.view(N, M * V * self.embedding_channels, T)
        x = self.data_bn(x)
        x = x.view(N, M, V, self.embedding_channels, T).permute(0, 1, 3, 4, 2).contiguous()
        x = x.view(N * M, self.embedding_channels, T, V)
        return x, N, M

    def extract_feature_map(self, x):
        """Input: (N, C, T, V, M) -> Output feature map: (N, D=512, T', V) trước pooling toàn cục.
        Dùng cho BodyPartGroundedConceptHeads (G1) — cắt lát theo chỉ số khớp NTU chuẩn."""
        x, N, M = self._embed_input(x)
        return self._backbone_layers(x, N, M)

    def extract_feat(self, x):
        """Input: (N, C, T, V, M) -> Output pooled feature: (N, 512)"""
        feature_map = self.extract_feature_map(x)
        feat = feature_map.mean(dim=(2, 3))
        feat = self.drop_out(feat)
        return feat

    def forward(self, x):
        feat = self.extract_feat(x)
        return self.fc(feat)





## ⚡ Block 4: Tầng Logic CRL (kế thừa nguyên trạng từ CLIR-SkateFormer)


In [ ]:
# 1. LiteralLayer & BinarizeSTE
"""LiteralLayer: Chuyển đổi concept logits thành literals nhị phân.

Module này thực hiện bước đầu tiên trong luận chứng logic (CRL pipeline):
  z^t ∈ R^(B × 72)  →  l^t ∈ R^(B × 144)

Hai chế độ:
  - 'soft': Dùng sigmoid liên tục (Phase 1 warm-up, gradient mượt)
  - 'ste':  Dùng Straight-Through Estimator nhị phân (Phase 2 rule sparsification)

Nguyên tắc bất biến:
  z^t (concept logits) LUÔN là số thực liên tục.
  Binarization CHỈ xảy ra khi tạo l^t để truyền vào CRL.
"""

import torch
import torch.nn as nn


class BinarizeSTE(torch.autograd.Function):
    """Straight-Through Estimator (STE) cho binarization.

    Forward:  hard = (z > 0).float()  [nhị phân {0, 1}]
    Backward: gradient đi thẳng qua sigmoid liên tục (không bị chặn bởi hard step)
    """

    @staticmethod
    def forward(ctx, z: torch.Tensor, temperature: float) -> torch.Tensor:
        # Lưu sigmoid liên tục để dùng trong backward
        soft = torch.sigmoid(temperature * z)
        ctx.save_for_backward(soft)
        ctx.temperature = temperature
        # Output nhị phân
        hard = (z > 0.0).float()
        # STE trick: hard + soft - soft.detach() → forward = hard, backward ≈ d(soft)/dz
        return hard + soft - soft.detach()

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        (soft,) = ctx.saved_tensors
        # Gradient của sigmoid: soft * (1 - soft) * temperature
        grad_z = grad_output * soft * (1.0 - soft) * ctx.temperature
        return grad_z, None  # None cho temperature (không học được)


def binarize_ste(z: torch.Tensor, temperature: float = 5.0) -> torch.Tensor:
    """Hàm tiện ích gọi BinarizeSTE."""
    return BinarizeSTE.apply(z, temperature)


class LiteralLayer(nn.Module):
    """Chuyển đổi concept logits z^t thành 144 literals.

    Output gồm 72 literal dương (p_i) nối với 72 literal phủ định (1 - p_i).

    Args:
        n_concepts: Số lượng concepts (mặc định 72).
        mode: 'soft' hoặc 'ste'.
        temperature: Nhiệt độ cho STE (annealing dần từ 5.0 → 10.0 ở Phase 2).
    """

    def __init__(self, n_concepts: int = 72, mode: str = "soft", temperature: float = 5.0):
        super().__init__()
        self.n_concepts = int(n_concepts)
        self._mode = mode
        self.temperature = float(temperature)

    @property
    def mode(self) -> str:
        return self._mode

    @mode.setter
    def mode(self, value: str):
        assert value in ("soft", "ste"), f"mode phải là 'soft' hoặc 'ste', nhận được: {value!r}"
        self._mode = value

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """Tính literals từ concept logits.

        Args:
            z: Concept logits [B, n_concepts]  (LUôN là số thực liên tục)

        Returns:
            l: Literals [B, 2 * n_concepts]  (cả dương lẫn phủ định)
        """
        if self._mode == "soft":
            p = torch.sigmoid(z)                   # [B, 74] ∈ (0, 1)
            l = torch.cat([p, 1.0 - p], dim=-1)   # [B, 148]
        else:  # 'ste'
            c_ste = binarize_ste(z, self.temperature)   # ≈ {0, 1}, grad qua sigmoid
            l = torch.cat([c_ste, 1.0 - c_ste], dim=-1)  # [B, 148]
        return l

    def extra_repr(self) -> str:
        return f"n_concepts={self.n_concepts}, mode={self._mode!r}, temperature={self.temperature}"




In [ ]:
# 2. ConjunctionLayer (L1 Part-Grouped & L2a AND2)
"""ConjunctionLayer: Tầng AND rules cho CRL pipeline.

Module này thực hiện hai tầng hội:
  L1: ConjunctionLayer  (AND, part-grouped) — 148 literals → 256 rules
  L2a: ConjunctionLayer2 (AND2, fully-connected) — 256 rules → 128 compound rules

Công thức Log-Space Product (chống vanishing gradient):
  r_j = exp( Σ_i w_ij · log(l_i + ε) )
       = ∏_i (l_i + ε)^w_ij

  Trong đó w_ij = sigmoid(θ_ij) ∈ [0, 1]  (trọng số học được)

Part-Grouped Architecture (L1):
  Mỗi nhóm rule chỉ kết nối với literals của nhóm bộ phận cơ thể tương ứng.
  Group mask là tensor nhị phân cố định (register_buffer).

  Nhóm bộ phận | Concepts (index) | Literals | Rules
  -------------|------------------|----------|------
  Head         |  0 – 5           |  12      |  32
  Hand         |  6 – 16          |  22      |  48
  Arm          | 17 – 27          |  22      |  48
  Hip          | 28 – 35          |  16      |  32
  Leg          | 36 – 44          |  18      |  32
  Foot         | 45 – 51          |  14      |  24
  Temporal     | 53 – 73          |  42      |  40
  Total        |                  | 148      | 256
"""

import torch
import torch.nn as nn

# ─── Hằng số cấu hình Part-Grouped ───────────────────────────────────────────

_DEFAULT_N_CONCEPTS = 72   # 72 concepts sau khi loại bỏ 2 null concepts (52 spatial + 20 temporal)
_DEFAULT_N_LITERALS = 144  # 2 × 72
_N_RULES_L1         = 256  # Σ rules của các nhóm
_N_RULES_L2A        = 128  # AND2 (L2a)

_EPS = 1e-6  # numerical stability cho log


def get_part_groups(n_concepts: int) -> list:
    """Trả về danh sách phân nhóm bộ phận cơ thể phù hợp với số concepts."""
    if n_concepts == 72:
        # Feeder mặc định remove_null_concept=True: 52 spatial + 20 temporal = 72
        return [
            ("head",     0,   6,   32),
            ("hand",     6,  17,   48),
            ("arm",     17,  28,   48),
            ("hip",     28,  36,   32),
            ("leg",     36,  45,   32),
            ("foot",    45,  52,   24),
            ("temporal",52,  72,   40),
        ]
    elif n_concepts == 74:
        # Full ontology chưa loại bỏ null concepts: 53 spatial + 21 temporal = 74
        return [
            ("head",     0,   6,   32),
            ("hand",     6,  17,   48),
            ("arm",     17,  28,   48),
            ("hip",     28,  36,   32),
            ("leg",     36,  45,   32),
            ("foot",    45,  53,   24),
            ("temporal",53,  74,   40),
        ]
    else:
        # Fallback chia đều cho các nhóm
        step = max(1, n_concepts // 7)
        return [
            ("group_0", 0, step, 32),
            ("group_1", step, 2 * step, 48),
            ("group_2", 2 * step, 3 * step, 48),
            ("group_3", 3 * step, 4 * step, 32),
            ("group_4", 4 * step, 5 * step, 32),
            ("group_5", 5 * step, 6 * step, 24),
            ("group_6", 6 * step, n_concepts, 40),
        ]


def _build_part_group_mask(n_literals: int = _DEFAULT_N_LITERALS, n_rules: int = _N_RULES_L1) -> torch.Tensor:
    """Xây dựng group mask nhị phân [n_literals, n_rules].

    Mask[i, j] = 1.0 nếu literal i thuộc nhóm bộ phận của rule j, 0.0 nếu không.
    Literals của concept c: index c (dương) và c + n_concepts (phủ định).
    """
    mask = torch.zeros(n_literals, n_rules)
    rule_start = 0
    n_concepts = n_literals // 2
    groups = get_part_groups(n_concepts)

    for (name, c_start, c_end, n_group_rules) in groups:
        rule_end = rule_start + n_group_rules
        for c in range(c_start, min(c_end, n_concepts)):
            # Literal dương: index c
            mask[c, rule_start:rule_end] = 1.0
            # Literal phủ định: index c + n_concepts
            mask[c + n_concepts, rule_start:rule_end] = 1.0
        rule_start = rule_end

    return mask


class ConjunctionLayer(nn.Module):
    """L1 AND layer với Part-Grouped structure.

    Input:  l ∈ R^(B × 148)   — literals từ LiteralLayer
    Output: r_and ∈ R^(B × 256) — conjunction rule activations

    Công thức:
        r_j = exp( Σ_i mask_ij · w_ij · log(l_i + ε) )

    Trọng số w_ij = sigmoid(θ_ij), và θ_ij bị zero-out ngoài mask
    (hiệu quả = không có kết nối ngoài nhóm).
    """

    def __init__(
        self,
        n_literals: int = _DEFAULT_N_LITERALS,
        n_rules: int = _N_RULES_L1,
        part_grouped: bool = True,
        eps: float = _EPS,
    ):
        super().__init__()
        self.n_literals = int(n_literals)
        self.n_rules = int(n_rules)
        self.part_grouped = bool(part_grouped)
        self.eps = float(eps)

        # Trọng số học được ở không gian logit
        self.weight_logit = nn.Parameter(
            torch.empty(n_literals, n_rules).fill_(-3.0)  # khởi tạo ≈ sigmoid(-3) ≈ 0.047 (thưa)
        )

        if part_grouped:
            mask = _build_part_group_mask(n_literals, n_rules)
            self.register_buffer("group_mask", mask)  # [148, 256]
        else:
            self.group_mask = None

    def _effective_weights(self) -> torch.Tensor:
        """Tính trọng số thực tế (có mask) w ∈ [0, 1]."""
        w = torch.sigmoid(self.weight_logit)  # [n_literals, n_rules]
        if self.group_mask is not None:
            w = w * self.group_mask            # zero-out ngoài nhóm
        return w

    def forward(self, l: torch.Tensor) -> torch.Tensor:
        """Tính conjunction rules.

        Args:
            l: Literals [B, n_literals] ∈ [0, 1]

        Returns:
            r_and: [B, n_rules] ∈ [0, 1]
        """
        w = self._effective_weights()   # [n_literals, n_rules]

        # log(l + ε): [B, n_literals]
        log_l = torch.log(l.clamp(min=self.eps))  # [B, n_literals]

        # Σ_i w_ij · log(l_i): einsum [B, n_lit] × [n_lit, n_rules] → [B, n_rules]
        log_r = torch.einsum("bi,ij->bj", log_l, w)  # [B, n_rules]

        # r_j = exp(log_r_j) ∈ (0, 1]
        return torch.exp(log_r)

    def sparsity_loss(self) -> torch.Tensor:
        """L1 penalty trên trọng số để ép rule thưa."""
        w = self._effective_weights()
        return w.mean()

    def extra_repr(self) -> str:
        return (
            f"n_literals={self.n_literals}, n_rules={self.n_rules}, "
            f"part_grouped={self.part_grouped}"
        )


class ConjunctionLayer2(nn.Module):
    """L2a AND2 layer: hội của các AND rules (fully-connected).

    Mô tả các hành động đòi hỏi nhiều bộ phận phối hợp đồng thời.
    Ví dụ: (arm_raise ∧ hand_lift) ∧ (head_tilt_up ∧ NOT leg_kick)

    Input:  r_and ∈ R^(B × 256)  — L1 conjunction rule activations
    Output: r_and2 ∈ R^(B × 128) — L2a compound conjunction rules

    Công thức:
        r_and2_k = exp( Σ_j w_jk · log(r_and_j + ε) )
    """

    def __init__(
        self,
        n_rules_l1: int = _N_RULES_L1,
        n_rules_l2a: int = _N_RULES_L2A,
        eps: float = _EPS,
    ):
        super().__init__()
        self.n_rules_l1 = int(n_rules_l1)
        self.n_rules_l2a = int(n_rules_l2a)
        self.eps = float(eps)

        # Khởi tạo thưa: sigmoid(-3) ≈ 0.047
        self.weight_logit = nn.Parameter(
            torch.empty(n_rules_l1, n_rules_l2a).fill_(-3.0)
        )

    def _effective_weights(self) -> torch.Tensor:
        return torch.sigmoid(self.weight_logit)  # [n_rules_l1, n_rules_l2a]

    def forward(self, r_and: torch.Tensor) -> torch.Tensor:
        """Tính L2a compound conjunction rules.

        Args:
            r_and: L1 rule activations [B, n_rules_l1] ∈ (0, 1]

        Returns:
            r_and2: [B, n_rules_l2a] ∈ (0, 1]
        """
        w = self._effective_weights()                    # [n_rules_l1, n_rules_l2a]
        log_r = torch.log(r_and.clamp(min=self.eps))    # [B, n_rules_l1]
        log_r2 = torch.einsum("bi,ij->bj", log_r, w)    # [B, n_rules_l2a]
        return torch.exp(log_r2)

    def sparsity_loss(self) -> torch.Tensor:
        return self._effective_weights().mean()

    def extra_repr(self) -> str:
        return f"n_rules_l1={self.n_rules_l1}, n_rules_l2a={self.n_rules_l2a}"




In [ ]:
# 3. DisjunctionLayer (L2b OR1)
"""DisjunctionLayer: Tầng OR rules cho CRL pipeline (L2b).

Module này thực hiện tuyển (disjunction) của các AND rules từ L1:
  r_and ∈ R^(B × 256)  →  r_or1 ∈ R^(B × 128)

Ý nghĩa: Mô tả các hành động có nhiều cách thực hiện khác nhau (alternative patterns).
Ví dụ: (arm_raise ∧ hand_lift) ∨ (hand_grasp ∧ arm_retract)

Công thức Probabilistic OR (De Morgan + log-space):
  r_or1_k = 1 - exp( Σ_j w_jk · log(1 - r_and_j + ε) )
           = 1 - ∏_j (1 - r_and_j + ε)^w_jk

  Trọng số w_jk = sigmoid(θ_jk) ∈ [0, 1]

Lưu ý:
  - Nếu tất cả w ≈ 0: r_or1 ≈ 0 (không kích hoạt)
  - Nếu ít nhất một AND rule j mạnh (r_and_j ≈ 1) và w_jk lớn:
    log(1 - r_and_j) → -∞  → r_or1_k → 1.0
"""

import torch
import torch.nn as nn

_EPS = 1e-6


class DisjunctionLayer(nn.Module):
    """L2b OR1 layer: tuyển của các AND rules (fully-connected).

    Input:  r_and  ∈ R^(B × n_rules_l1)  — L1 conjunction rule activations
    Output: r_or1  ∈ R^(B × n_rules_l2b) — L2b disjunction rule activations

    Công thức:
        r_or1_k = 1 - exp( Σ_j w_jk · log(1 - r_and_j + ε) )

    Trong đó w_jk = sigmoid(θ_jk), với θ_jk là trọng số học được.
    """

    def __init__(
        self,
        n_rules_l1: int = 256,
        n_rules_l2b: int = 128,
        eps: float = _EPS,
    ):
        super().__init__()
        self.n_rules_l1  = int(n_rules_l1)
        self.n_rules_l2b = int(n_rules_l2b)
        self.eps = float(eps)

        # Khởi tạo thưa: sigmoid(-3) ≈ 0.047
        self.weight_logit = nn.Parameter(
            torch.empty(n_rules_l1, n_rules_l2b).fill_(-3.0)
        )

    def _effective_weights(self) -> torch.Tensor:
        """Trọng số thực tế w ∈ [0, 1]."""
        return torch.sigmoid(self.weight_logit)  # [n_rules_l1, n_rules_l2b]

    def forward(self, r_and: torch.Tensor) -> torch.Tensor:
        """Tính disjunction rules.

        Args:
            r_and: L1 conjunction activations [B, n_rules_l1] ∈ (0, 1]

        Returns:
            r_or1: [B, n_rules_l2b] ∈ [0, 1)
        """
        w = self._effective_weights()   # [n_rules_l1, n_rules_l2b]

        # log(1 - r_and + ε): phần bù của AND rule
        log_complement = torch.log((1.0 - r_and).clamp(min=self.eps))  # [B, n_rules_l1]

        # Σ_j w_jk · log(1 - r_and_j): log-space weighted sum
        log_prod = torch.einsum("bi,ij->bj", log_complement, w)   # [B, n_rules_l2b]

        # r_or1 = 1 - exp(log_prod)
        r_or1 = 1.0 - torch.exp(log_prod)   # [B, n_rules_l2b] ∈ [0, 1)
        return r_or1

    def sparsity_loss(self) -> torch.Tensor:
        """L1 penalty trên trọng số để ép rule thưa."""
        return self._effective_weights().mean()

    def extra_repr(self) -> str:
        return f"n_rules_l1={self.n_rules_l1}, n_rules_l2b={self.n_rules_l2b}"




In [ ]:
# 4. CRLReasoner (1-Pass Shared Reasoner)
"""CRLReasoner: Concept Reasoning Logic Reasoner.

Module này gói gọn toàn bộ pipeline suy luận logic 1 lượt:
  z^t ∈ R^(B × 72)
    → LiteralLayer → l^t ∈ R^(B × 144)
    → ConjunctionLayer (L1 AND, Part-grouped) → r_and ∈ R^(B × 256)
    → [ConjunctionLayer2 (L2a AND2) ‖ DisjunctionLayer (L2b OR1)]
    → r_and2 ∈ R^(B × 128), r_or1 ∈ R^(B × 128)
    → ActionHead [Linear: 256 → 120] → y^t ∈ R^(B × 120)

    Logic Evidence (cho ConceptUpdateCell):
      r^t = [r_and; r_and2; r_or1] ∈ R^(B × 512)

Trọng số được chia sẻ (shared) giữa các vòng lặp trong IterativeLogicReasoner.

Thiết kế:
  - ActionHead KHÔNG nhận feature bypass từ backbone
    (đảm bảo tính giải thích tuyệt đối — strict logic mode)
  - Logic Evidence kết hợp cả ba tầng: L1 (body-part features),
    L2a (coordinated patterns), L2b (alternative patterns)
"""

from typing import Tuple

import torch
import torch.nn as nn

# relative import in notebook scope
# relative import in notebook scope
# relative import in notebook scope

# ─── Kích thước mặc định ────────────────────────────────────────────────────
_N_CONCEPTS  = 72   # số concepts (72 sau khi loại 2 null concepts)
_N_ACTIONS   = 120  # số lớp hành động NTU RGB+D 120
_R1          = 256  # L1 AND rules
_R2A         = 128  # L2a AND2 rules
_R2B         = 128  # L2b OR1 rules
_LOGIC_DIM   = _R1 + _R2A + _R2B  # = 512


class CRLReasoner(nn.Module):
    """Concept Reasoning Logic Reasoner (1-pass).

    Được dùng với shared weights trong IterativeLogicReasoner:
    mỗi vòng lặp gọi cùng 1 instance CRLReasoner này.

    Args:
        n_concepts:  Số concepts (mặc định 72).
        n_actions:   Số lớp hành động (mặc định 120).
        r1:          Số L1 AND rules (mặc định 256).
        r2a:         Số L2a AND2 rules (mặc định 128).
        r2b:         Số L2b OR1 rules (mặc định 128).
        literal_mode: 'soft' hoặc 'ste' (có thể thay đổi runtime).
        temperature: Nhiệt độ STE cho LiteralLayer.
    """

    def __init__(
        self,
        n_concepts:   int   = _N_CONCEPTS,
        n_actions:    int   = _N_ACTIONS,
        r1:           int   = _R1,
        r2a:          int   = _R2A,
        r2b:          int   = _R2B,
        literal_mode: str   = "soft",
        temperature:  float = 5.0,
    ):
        super().__init__()
        self.n_concepts  = int(n_concepts)
        self.n_actions   = int(n_actions)
        self.r1  = int(r1)
        self.r2a = int(r2a)
        self.r2b = int(r2b)
        self.logic_dim = r1 + r2a + r2b  # = 512 mặc định

        # ── Các sub-module ──────────────────────────────────────────────────
        self.literal_layer = LiteralLayer(
            n_concepts=n_concepts,
            mode=literal_mode,
            temperature=temperature,
        )

        self.l1_and   = ConjunctionLayer(
            n_literals=2 * n_concepts,  # 148
            n_rules=r1,                 # 256
            part_grouped=True,
        )

        self.l2a_and2 = ConjunctionLayer2(
            n_rules_l1=r1,    # 256
            n_rules_l2a=r2a,  # 128
        )

        self.l2b_or1  = DisjunctionLayer(
            n_rules_l1=r1,    # 256
            n_rules_l2b=r2b,  # 128
        )

        # ActionHead: KHÔNG có feature bypass — strict logic mode
        self.action_head = nn.Linear(r2a + r2b, n_actions)  # 256 → 120

    # ── Properties để điều chỉnh LiteralLayer ────────────────────────────────
    @property
    def literal_mode(self) -> str:
        return self.literal_layer.mode

    @literal_mode.setter
    def literal_mode(self, value: str):
        self.literal_layer.mode = value

    @property
    def temperature(self) -> float:
        return self.literal_layer.temperature

    @temperature.setter
    def temperature(self, value: float):
        self.literal_layer.temperature = float(value)

    # ── Forward ──────────────────────────────────────────────────────────────
    def forward(
        self,
        z: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Suy luận logic 1 lượt.

        Args:
            z: Concept logits [B, n_concepts]  (LUôN là số thực liên tục)

        Returns:
            r_evidence: Logic Evidence [B, logic_dim]   (= [r_and; r_and2; r_or1])
            y:          Action logits  [B, n_actions]
        """
        # Bước 1: Tạo literals (chỉ binarize ở đây, z vẫn liên tục)
        l = self.literal_layer(z)              # [B, 148]

        # Bước 2: L1 AND rules (part-grouped)
        r_and = self.l1_and(l)                 # [B, 256]

        # Bước 3: L2 song song
        r_and2 = self.l2a_and2(r_and)         # [B, 128]
        r_or1  = self.l2b_or1(r_and)          # [B, 128]

        # Bước 4: ActionHead (strict logic mode)
        l2_cat = torch.cat([r_and2, r_or1], dim=-1)  # [B, 256]
        y = self.action_head(l2_cat)                   # [B, n_actions]

        # Logic Evidence: [r_and; r_and2; r_or1] = [B, 256+128+128] = [B, 512]
        r_evidence = torch.cat([r_and, r_and2, r_or1], dim=-1)

        return r_evidence, y

    # ── Loss helpers ─────────────────────────────────────────────────────────
    def rule_sparsity_loss(self) -> torch.Tensor:
        """Rule Sparsity Loss theo kế hoạch (Mục 8.2 CLIR plan).

        L_sparse = mean(w_and1) + mean(w_and2) + mean(w_or1)
        """
        return (
            self.l1_and.sparsity_loss()
            + self.l2a_and2.sparsity_loss()
            + self.l2b_or1.sparsity_loss()
        ) / 3.0

    def extra_repr(self) -> str:
        return (
            f"n_concepts={self.n_concepts}, n_actions={self.n_actions}, "
            f"r1={self.r1}, r2a={self.r2a}, r2b={self.r2b}, "
            f"logic_dim={self.logic_dim}"
        )




## 🌐 Block 5: G2 — LightConceptGrounding (CLIP đóng băng)
CLIP **không** LoRA fine-tune — chỉ encode text anchor 1 lần lúc khởi tạo model, học 1 projection nhỏ. Nếu không có internet/không tải được CLIP checkpoint, module tự tắt (`available=False`, `L_ground=0`) — model vẫn chạy bình thường.


In [ ]:
# LightConceptGrounding (G2)
"""LightConceptGrounding: Semantic grounding nhẹ cho concept (G2 trong
`Architecture/10_CLIR-SkateFormer_v2_Proposal.md`).

Khác với REASON/GRAFT-SAR (LoRA fine-tune toàn bộ CLIP text encoder + prompt
theo từng batch), module này:
  - Dùng CLIP **đóng băng hoàn toàn** (không LoRA, không backprop qua text tower).
  - Encode text anchor cho từng concept **đúng 1 lần** lúc khởi tạo model, lưu vào
    buffer — không forward CLIP lại trong training loop.
  - Chỉ học một projection nhỏ (Linear + LayerNorm) từ concept feature (đã có sẵn
    từ BodyPartGroundedConceptHeads) sang không gian CLIP.

An toàn khi không có internet / không tải được checkpoint CLIP: `available=False`,
mọi lời gọi loss trả về 0 thay vì crash — giữ nguyên tắc "graceful fallback" như
`_load_backbone_weights` của CLIR-SkateFormer.
"""

from __future__ import annotations

from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


def default_concept_prompt(concept_name: str) -> str:
    """Sinh câu mô tả mặc định từ tên concept dạng snake_case, vd:
    'hand_lift_to_face' -> 'a person is doing hand lift to face'."""
    words = concept_name.replace("_", " ")
    return f"a person is doing {words}"


class LightConceptGrounding(nn.Module):
    def __init__(
        self,
        feature_dim: int,
        concept_names: List[str],
        clip_backbone: str = "ViT-B/32",
        clip_dim: int = 512,
        device: Optional[torch.device] = None,
        concept_prompts: Optional[List[str]] = None,
    ):
        super().__init__()
        self.concept_names = list(concept_names)
        self.num_concepts = len(self.concept_names)
        self.clip_dim = int(clip_dim)

        self.feat_proj = nn.Sequential(
            nn.Linear(feature_dim, clip_dim),
            nn.LayerNorm(clip_dim),
        )

        self.register_buffer("text_anchors", torch.zeros(self.num_concepts, clip_dim))
        self.available = False

        prompts = concept_prompts or [default_concept_prompt(c) for c in self.concept_names]
        self._try_build_text_anchors(prompts, clip_backbone, device)

    @torch.no_grad()
    def _try_build_text_anchors(self, prompts: List[str], clip_backbone: str, device):
        try:
            import clip  # cai qua pip trong Block 1 (git+https://github.com/openai/CLIP.git)
        except Exception as e:
            print(f"[LightConceptGrounding] Khong import duoc goi 'clip': {e}. Grounding se bi TAT.")
            return
        try:
            device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
            clip_model, _ = clip.load(clip_backbone, device=device)
            clip_model.eval()
            tokens = clip.tokenize(prompts, truncate=True).to(device)
            text_features = clip_model.encode_text(tokens).float()
            text_features = F.normalize(text_features, dim=-1).cpu()
            self.text_anchors.copy_(text_features)
            self.available = True
            del clip_model
            print(f"[LightConceptGrounding] Da encode {len(prompts)} concept text anchors qua CLIP '{clip_backbone}' (dong bang).")
        except Exception as e:
            print(f"[LightConceptGrounding] Khong tai duoc CLIP checkpoint ({e}). Grounding se bi TAT — "
                  f"model van chay binh thuong, chi thieu L_ground.")

    def project(self, feat: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.feat_proj(feat), dim=-1)

    def compute_loss(
        self,
        group_feats: Dict[str, torch.Tensor],
        concept_targets: torch.Tensor,
        concept_slices: Dict[str, Tuple[int, int]],
        temperature: float = 0.07,
    ) -> torch.Tensor:
        """Multi-positive InfoNCE giữa skeleton feature từng nhóm và text anchor của
        đúng các concept thuộc nhóm đó.

        Args:
            group_feats: {group_name: [B, feature_dim]}
            concept_targets: [B, num_concepts] nhãn nhị phân
            concept_slices: {group_name: (start, end)} chỉ số concept của nhóm trong 72 concept
        """
        if not self.available or not group_feats:
            ref = next(iter(group_feats.values())) if group_feats else concept_targets
            return ref.new_zeros(())

        device = concept_targets.device
        total_loss = concept_targets.new_zeros(())
        count = 0
        for g, feat in group_feats.items():
            if g not in concept_slices:
                continue
            start, end = concept_slices[g]
            if end <= start:
                continue
            pos_mask = concept_targets[:, start:end].to(device)  # [B, C_g]
            if pos_mask.sum() == 0:
                continue

            skel_embed = self.project(feat)                              # [B, clip_dim]
            text_embed = self.text_anchors[start:end].to(device)         # [C_g, clip_dim]
            logits = skel_embed @ text_embed.t() / temperature           # [B, C_g]
            log_prob = F.log_softmax(logits, dim=-1)

            pos_count = pos_mask.sum(dim=-1, keepdim=True).clamp_min(1.0)
            loss_g = -(log_prob * pos_mask).sum(dim=-1) / pos_count.squeeze(-1)  # [B]
            total_loss = total_loss + loss_g.mean()
            count += 1

        if count == 0:
            return concept_targets.new_zeros(())
        return total_loss / count

    def extra_repr(self) -> str:
        return f"num_concepts={self.num_concepts}, clip_dim={self.clip_dim}, available={self.available}"




## 🔄 Block 6: ConceptUpdateCell, IterativeLogicReasoner, BodyPartGroundedConceptHeads (G1) & GLIRHyperGCN


In [ ]:
# GLIR-HyperGCN Main Components
"""GLIR-HyperGCN: Grounded Logic Iterative Refinement với backbone HyperGCN.

Nâng cấp trực tiếp từ CLIR-SkateFormer (`d:\\Thesis\\CLIR-SkateFormer\\model\\clir_skateformer.py`)
theo `Architecture/10_CLIR-SkateFormer_v2_Proposal.md`, đổi backbone sang HyperGCN
(`d:\\Thesis\\Hyper-GCN`, bản làm sạch tại `model/hypergcn.py`) và triển khai:

  G1 — BodyPartGroundedConceptHeads: concept không gian được dự đoán RIÊNG cho từng
       nhóm khớp NTU-25 (Head/Hand/Arm/Hip/Leg/Foot), khớp 1-1 với part-grouped L1
       ở tầng logic (`crl/conjunction_layer.py::get_part_groups`) — khắc phục khoảng
       trống G1 (concept head cũ dùng chung 1 vector pooled toàn cục cho mọi concept).
  G2 — LightConceptGrounding: CLIP đóng băng, chỉ học 1 projection nhỏ, text anchor
       encode 1 lần lúc khởi tạo (`crl/light_grounding.py`).
  G3 — set_progress(): lịch annealing liên tục (temperature/beta_max/lambda_reg theo
       progress ∈ [0,1]) thay cho set_phase() rời rạc — set_phase() vẫn giữ lại để
       chạy cấu hình V2 (chưa bật G3); set_progress() dùng cho cấu hình V3.

Vòng lặp logic (ConceptUpdateCell + IterativeLogicReasoner) và hệ loss (Asymmetric
BCE, rule sparsity, loop stability) giữ NGUYÊN từ CLIR-SkateFormer — cơ chế này
backbone-agnostic, không cần thay đổi khi đổi từ SkateFormer sang HyperGCN.
"""

from __future__ import annotations

from collections import OrderedDict
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import torch
import torch.nn as nn
import torch.nn.functional as F

# relative import in notebook scope
# relative import in notebook scope
# relative import in notebook scope
# relative import in notebook scope

# ─── Hằng số mặc định ────────────────────────────────────────────────────────

_N_CONCEPTS_SPATIAL  = 52
_N_CONCEPTS_TEMPORAL = 20
_N_CONCEPTS          = 72
_N_ACTIONS           = 120
_LOGIC_DIM           = 512

# NTU 25-joint chuẩn (0-indexed), nhất quán với HyperGraph.inward_ori_index trong
# model/hypergcn.py (1-indexed trong file gốc, đã trừ 1). Phủ đủ 25 khớp, không
# chồng lấn. Thứ tự nhóm PHẢI khớp với get_part_groups() ở conjunction_layer.py
# (head, hand, arm, hip, leg, foot) để concept đầu ra đúng slice.
NTU25_BODY_PART_JOINTS: Dict[str, List[int]] = {
    "head": [2, 3, 20],                       # Neck, Head, SpineShoulder
    "hand": [6, 7, 10, 11, 21, 22, 23, 24],    # WristL/R, HandL/R, HandTipL/R, ThumbL/R
    "arm":  [4, 5, 8, 9],                      # ShoulderL/R, ElbowL/R
    "hip":  [0, 1, 12, 16],                    # SpineBase, SpineMid, HipL, HipR
    "leg":  [13, 17],                          # KneeL, KneeR
    "foot": [14, 15, 18, 19],                  # AnkleL/R, FootL/R
}


# ─────────────────────────────────────────────────────────────────────────────
# ConceptUpdateCell / IterativeLogicReasoner — kế thừa nguyên trạng từ CLIR
# ─────────────────────────────────────────────────────────────────────────────

class ConceptUpdateCell(nn.Module):
    """Logic-Guided Concept Update Cell (Top-Down Feedback). Giống hệt CLIR-SkateFormer —
    backbone-agnostic, chỉ nhận z_t (concept logits) và r_t (logic evidence)."""

    def __init__(
        self,
        num_concepts: int = _N_CONCEPTS,
        logic_dim:    int = _LOGIC_DIM,
        hidden_dim:   int = 256,
        dropout:      float = 0.1,
    ):
        super().__init__()
        self.num_concepts = int(num_concepts)
        self.logic_dim    = int(logic_dim)

        self.logic_proj = nn.Sequential(
            nn.Linear(logic_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, num_concepts),
        )
        self.update_mlp = nn.Sequential(
            nn.Linear(2 * num_concepts, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_concepts),
        )

    def forward(self, z_t: torch.Tensor, r_t: torch.Tensor) -> torch.Tensor:
        r_proj = self.logic_proj(r_t)
        cell_input = torch.cat([z_t, r_proj], dim=-1)
        return self.update_mlp(cell_input)


class IterativeLogicReasoner(nn.Module):
    """Vòng lặp suy luận lặp K với Shared CRLReasoner. Giống hệt CLIR-SkateFormer."""

    def __init__(
        self,
        num_concepts: int   = _N_CONCEPTS,
        num_actions:  int   = _N_ACTIONS,
        num_loops:    int   = 2,
        r1:           int   = 256,
        r2a:          int   = 128,
        r2b:          int   = 128,
        beta_max:     float = 0.5,
        beta_init:    float = 0.1,
        literal_mode: str   = "soft",
        temperature:  float = 5.0,
        hidden_dim:   int   = 256,
        dropout:      float = 0.1,
        random_feedback: bool = False,
    ):
        super().__init__()
        self.num_concepts = int(num_concepts)
        self.num_actions  = int(num_actions)
        self.num_loops    = max(0, int(num_loops))
        self.beta_max     = float(beta_max)
        self.random_feedback = bool(random_feedback)
        logic_dim = r1 + r2a + r2b

        self.crl = CRLReasoner(
            n_concepts=num_concepts, n_actions=num_actions,
            r1=r1, r2a=r2a, r2b=r2b,
            literal_mode=literal_mode, temperature=temperature,
        )
        self.update_cell = ConceptUpdateCell(
            num_concepts=num_concepts, logic_dim=logic_dim,
            hidden_dim=hidden_dim, dropout=dropout,
        )

        if self.num_loops == 0:
            self.raw_beta = nn.Parameter(torch.empty(0, dtype=torch.float32))
        elif self.beta_max <= 0.0:
            raw_init = torch.tensor(-10.0, dtype=torch.float32)
            self.raw_beta = nn.Parameter(raw_init.unsqueeze(0).expand(self.num_loops).clone())
        else:
            ratio = min(max(beta_init / self.beta_max, 1e-4), 1.0 - 1e-4)
            raw_init = torch.logit(torch.tensor(ratio, dtype=torch.float32))
            self.raw_beta = nn.Parameter(raw_init.unsqueeze(0).expand(self.num_loops).clone())

    @property
    def literal_mode(self) -> str:
        return self.crl.literal_mode

    @literal_mode.setter
    def literal_mode(self, value: str):
        self.crl.literal_mode = value

    @property
    def temperature(self) -> float:
        return self.crl.temperature

    @temperature.setter
    def temperature(self, value: float):
        self.crl.temperature = float(value)

    def get_betas(self) -> torch.Tensor:
        if self.num_loops == 0:
            return self.raw_beta
        if self.beta_max <= 0.0:
            return torch.zeros_like(self.raw_beta)
        return torch.sigmoid(self.raw_beta) * self.beta_max

    def forward(self, z0: torch.Tensor, return_intermediates: bool = True) -> Dict:
        betas = self.get_betas()
        z_t = z0
        z_all = [z0] if return_intermediates else []
        y_all = []

        for t in range(self.num_loops):
            r_t, y_t = self.crl(z_t)
            y_all.append(y_t)
            feedback_r = torch.rand_like(r_t) if self.random_feedback else r_t
            delta_t = self.update_cell(z_t, feedback_r)
            z_t = z_t + betas[t] * delta_t
            if return_intermediates:
                z_all.append(z_t)

        _, y_final = self.crl(z_t)
        return {"y_final": y_final, "z_final": z_t, "z_all": z_all, "y_all": y_all, "betas": betas}

    def rule_sparsity_loss(self) -> torch.Tensor:
        return self.crl.rule_sparsity_loss()


# ─────────────────────────────────────────────────────────────────────────────
# G1: BodyPartGroundedConceptHeads
# ─────────────────────────────────────────────────────────────────────────────

class BodyPartGroundedConceptHeads(nn.Module):
    """Mỗi nhóm bộ phận pool RIÊNG trên đúng tập khớp NTU-25 của mình rồi mới suy
    ra concept của nhóm đó — khớp 1-1 với part_groups ở tầng logic (get_part_groups)."""

    def __init__(self, feature_dim: int, spatial_concepts: int = _N_CONCEPTS_SPATIAL, dropout: float = 0.1):
        super().__init__()
        all_groups = get_part_groups(spatial_concepts + _N_CONCEPTS_TEMPORAL)
        spatial_groups = [g for g in all_groups if g[0] != "temporal"]

        self.group_order = [g[0] for g in spatial_groups]
        self.concept_slices: Dict[str, Tuple[int, int]] = {g[0]: (g[1], g[2]) for g in spatial_groups}

        missing = set(self.group_order) - set(NTU25_BODY_PART_JOINTS)
        if missing:
            raise ValueError(f"NTU25_BODY_PART_JOINTS thieu nhom: {missing}")

        total = sum(end - start for start, end in self.concept_slices.values())
        if total != spatial_concepts:
            raise ValueError(f"Tong concept theo nhom ({total}) != spatial_concepts ({spatial_concepts})")

        self.pool_scores = nn.ModuleDict({
            g: nn.Conv1d(feature_dim, 1, kernel_size=1) for g in self.group_order
        })
        self.heads = nn.ModuleDict({
            g: nn.Sequential(
                nn.Linear(feature_dim, 128), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(128, self.concept_slices[g][1] - self.concept_slices[g][0]),
            ) for g in self.group_order
        })

    def forward(self, feature_map: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """feature_map: [B, D, T', V] (V=25, ánh xạ 1:1 khớp NTU chuẩn)."""
        outs = []
        group_feats: Dict[str, torch.Tensor] = {}
        for g in self.group_order:
            joint_idx = NTU25_BODY_PART_JOINTS[g]
            sub = feature_map[:, :, :, joint_idx].mean(dim=2)            # [B, D, |joints_g|] (pool theo T')
            attn = torch.softmax(self.pool_scores[g](sub), dim=-1)       # [B, 1, |joints_g|]
            pooled = (sub * attn).sum(dim=-1)                            # [B, D] — CHỈ từ khớp thuộc nhóm g
            group_feats[g] = pooled
            outs.append(self.heads[g](pooled))
        z_spatial = torch.cat(outs, dim=-1)
        return z_spatial, group_feats


# ─────────────────────────────────────────────────────────────────────────────
# GLIRHyperGCN
# ─────────────────────────────────────────────────────────────────────────────

class GLIRHyperGCN(nn.Module):
    def __init__(
        self,
        num_classes:        int   = _N_ACTIONS,
        num_concepts:        int   = _N_CONCEPTS,
        spatial_concepts:    int   = _N_CONCEPTS_SPATIAL,
        temporal_concepts:   int   = _N_CONCEPTS_TEMPORAL,
        num_person:          int   = 2,
        num_point:           int   = 25,
        in_channels:         int   = 3,
        embedding_channels:  int   = 128,
        backbone_weights:    Optional[str]       = None,
        concept_names:       Optional[List[str]] = None,
        # Reasoner
        num_loops:    int   = 2,
        r1:           int   = 256,
        r2a:          int   = 128,
        r2b:          int   = 128,
        beta_max:     float = 0.5,
        beta_init:    float = 0.1,
        literal_mode: str   = "soft",
        temperature:  float = 5.0,
        hidden_dim:   int   = 256,
        dropout:      float = 0.1,
        # Loss weights
        lambda_sup:              float = 1.0,
        lambda_reg:              float = 0.0,
        lambda_ground:            float = 0.1,
        concept_negative_weight: float = 0.15,
        concept_positive_weight: float = 1.0,
        asymmetric_gamma_neg:    float = 2.0,
        return_aux:              bool  = True,
        # G2 — grounding
        use_grounding:      bool = True,
        clip_backbone:       str = "ViT-B/32",
        # Ablations (kế thừa từ CLIR)
        random_feedback:    bool = False,
        **kwargs,
    ):
        super().__init__()
        if spatial_concepts + temporal_concepts != num_concepts:
            raise ValueError(
                f"spatial_concepts({spatial_concepts}) + temporal_concepts({temporal_concepts})"
                f" != num_concepts({num_concepts})"
            )

        self.num_classes        = int(num_classes)
        self.num_concepts       = int(num_concepts)
        self.spatial_concepts   = int(spatial_concepts)
        self.temporal_concepts  = int(temporal_concepts)
        self.num_loops          = max(0, int(num_loops))
        self.lambda_sup         = float(lambda_sup)
        self.lambda_reg         = float(lambda_reg)
        self.lambda_ground       = float(lambda_ground)
        self.concept_neg_weight = float(concept_negative_weight)
        self.concept_pos_weight = float(concept_positive_weight)
        self.gamma_neg          = float(asymmetric_gamma_neg)
        self.return_aux         = bool(return_aux)
        self.accepts_concepts   = True
        self._progress          = 0.0

        # ── Backbone: HyperGCN ────────────────────────────────────────────────
        self.backbone = HyperGCNModel(
            num_class=num_classes, num_point=num_point, num_person=num_person,
            in_channels=in_channels, embedding_channels=embedding_channels,
            drop_out=dropout,
        )
        self._load_backbone_weights(backbone_weights)
        feature_dim = self.backbone.feat_dim

        # ── G1: Concept Heads theo bộ phận cơ thể ─────────────────────────────
        self.concept_heads = BodyPartGroundedConceptHeads(feature_dim, spatial_concepts, dropout)
        self.temporal_pool_score = nn.Conv1d(feature_dim, 1, kernel_size=1)
        self.temporal_head = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, temporal_concepts),
        )

        if concept_names is None:
            concept_names = [f"concept_{i}" for i in range(self.num_concepts)]
        self.concept_names = list(concept_names)

        # ── G2: Semantic Grounding (nhẹ, CLIP đóng băng) ──────────────────────
        self.use_grounding = bool(use_grounding)
        if self.use_grounding:
            self.grounding = LightConceptGrounding(
                feature_dim=feature_dim, concept_names=self.concept_names,
                clip_backbone=clip_backbone,
            )
        else:
            self.grounding = None

        # ── IterativeLogicReasoner (kế thừa CLIR nguyên trạng) ────────────────
        self.reasoner = IterativeLogicReasoner(
            num_concepts=self.num_concepts, num_actions=self.num_classes,
            num_loops=self.num_loops, r1=r1, r2a=r2a, r2b=r2b,
            beta_max=beta_max, beta_init=beta_init,
            literal_mode=literal_mode, temperature=temperature,
            hidden_dim=hidden_dim, dropout=dropout,
            random_feedback=random_feedback,
        )

    # ── Backbone weight loading (giống pattern của CLIR) ──────────────────────

    def _load_backbone_weights(self, checkpoint_path: Optional[str]):
        if not checkpoint_path:
            return
        path = Path(checkpoint_path)
        if not path.exists():
            print(f"[Warning] HyperGCN checkpoint khong ton tai: {path}. Khoi tao ngau nhien.")
            return
        checkpoint = torch.load(str(path), map_location="cpu")
        if isinstance(checkpoint, dict):
            for key in ("state_dict", "model"):
                if key in checkpoint and isinstance(checkpoint[key], dict):
                    checkpoint = checkpoint[key]
                    break
        if not isinstance(checkpoint, dict):
            raise TypeError("Dinh dang checkpoint HyperGCN khong ho tro")

        backbone_state = self.backbone.state_dict()
        mapped = OrderedDict()
        for key, value in checkpoint.items():
            clean_key = key.replace("module.", "", 1)
            if clean_key.startswith("backbone."):
                clean_key = clean_key[len("backbone."):]
            if clean_key in backbone_state and backbone_state[clean_key].shape == value.shape:
                mapped[clean_key] = value
        missing, unexpected = self.backbone.load_state_dict(mapped, strict=False)
        print(f"Loaded {len(mapped)} HyperGCN tensors tu {path}; missing={len(missing)}, unexpected={len(unexpected)}")

    # ── Pooling temporal (giữ nguyên cơ chế cũ — concept thời gian không thuộc riêng bộ phận nào) ──

    def _pool_temporal(self, feature_map: torch.Tensor) -> torch.Tensor:
        temporal_tokens = feature_map.mean(dim=3)                                    # [B, D, T']
        attn = torch.softmax(self.temporal_pool_score(temporal_tokens), dim=-1)
        return (temporal_tokens * attn).sum(dim=-1)                                  # [B, D]

    # ── G3: lịch huấn luyện ────────────────────────────────────────────────────

    def set_phase(self, phase: int, temperature: Optional[float] = None):
        """Giữ nguyên cơ chế 2-phase rời rạc của CLIR — dùng cho cấu hình V2 (G1+G2, CHƯA G3)."""
        if phase == 1:
            self.reasoner.literal_mode = "soft"
            if self.reasoner.num_loops > 0 and self.reasoner.beta_max > 0.0:
                self.reasoner.beta_max = 0.1
                with torch.no_grad():
                    self.reasoner.raw_beta.fill_(float(torch.logit(torch.tensor(0.95))))
            self.lambda_reg = 0.0
            if temperature is not None:
                self.reasoner.temperature = temperature
        elif phase == 2:
            self.reasoner.literal_mode = "ste"
            if self.reasoner.num_loops > 0 and self.reasoner.beta_max > 0.0:
                old_beta_max = self.reasoner.beta_max
                new_beta_max = 0.5
                with torch.no_grad():
                    current_beta = torch.sigmoid(self.reasoner.raw_beta) * old_beta_max
                    new_ratio = (current_beta / new_beta_max).clamp(1e-4, 1.0 - 1e-4)
                    self.reasoner.raw_beta.copy_(torch.logit(new_ratio))
                self.reasoner.beta_max = new_beta_max
            self.lambda_reg = 0.005
            if temperature is not None:
                self.reasoner.temperature = temperature
        else:
            raise ValueError(f"phase phai la 1 hoac 2, nhan duoc: {phase}")

    def set_progress(self, progress: float):
        """G3 — annealing lien tuc thay set_phase() roi rac. Dung cho cau hinh V3.

        temperature: 1.0 -> 10.0, beta_max: 0.1 -> 0.5, lambda_reg: 0 -> 0.005
        (bat dau tu progress=0.3), khong doi lai optimizer param-group giua chung.
        """
        progress = float(max(0.0, min(1.0, progress)))
        self._progress = progress
        self.reasoner.literal_mode = "ste"
        self.reasoner.temperature = 1.0 + progress * (10.0 - 1.0)
        if self.reasoner.num_loops > 0 and self.reasoner.beta_max is not None:
            self.reasoner.beta_max = 0.1 + progress * (0.5 - 0.1)
            with torch.no_grad():
                self.reasoner.raw_beta.fill_(float(torch.logit(torch.tensor(0.95))))
        self.lambda_reg = 0.0 if progress < 0.3 else 0.005 * (progress - 0.3) / 0.7

    # ── Loss Functions (Asymmetric BCE — giữ nguyên từ CLIR) ──────────────────

    def _asymmetric_bce(self, logits, targets, mask=None) -> torch.Tensor:
        targets = targets.float().clamp(0.0, 1.0)
        probs = torch.sigmoid(logits)
        pos_loss = F.softplus(-logits) * targets
        neg_loss = F.softplus(logits) * (1.0 - probs).pow(self.gamma_neg) * (1.0 - targets)
        weights = targets * self.concept_pos_weight + (1.0 - targets) * self.concept_neg_weight
        if mask is not None:
            weights = weights * mask.float()
        loss = pos_loss + neg_loss
        denom = weights.sum().clamp_min(1.0)
        return (loss * weights).sum() / denom

    def compute_sup_loss(self, z0, z_all, y_all, concept_targets, action_labels, concept_mask=None) -> Dict[str, torch.Tensor]:
        z_K = z_all[-1]
        l_c_K = self._asymmetric_bce(z_K, concept_targets, mask=concept_mask)
        l_c_0 = self._asymmetric_bce(z0, concept_targets, mask=concept_mask)
        l_act_aux = z0.new_zeros(())
        if len(y_all) > 0:
            for y_t in y_all:
                l_act_aux = l_act_aux + F.cross_entropy(y_t, action_labels)
            l_act_aux = l_act_aux / len(y_all)
        l_sup = l_c_K + 0.5 * l_c_0 + 0.3 * l_act_aux
        return {"l_c_K": l_c_K, "l_c_0": l_c_0, "l_act_aux": l_act_aux, "l_sup": l_sup}

    def compute_reg_loss(self, z_all) -> Dict[str, torch.Tensor]:
        l_sparse = self.reasoner.rule_sparsity_loss()
        l_stable = z_all[0].new_zeros(())
        K = len(z_all) - 1
        if K > 0:
            for t in range(K):
                diff = (torch.sigmoid(z_all[t + 1]) - torch.sigmoid(z_all[t])).abs()
                l_stable = l_stable + diff.mean()
            l_stable = l_stable / K
        l_reg = l_sparse + l_stable
        return {"l_sparse": l_sparse, "l_stable": l_stable, "l_reg": l_reg}

    def compute_grounding_loss(self, group_feats: Dict[str, torch.Tensor], concept_targets: torch.Tensor) -> torch.Tensor:
        """G2 — chỉ tính khi use_grounding=True và CLIP tai duoc thanh cong."""
        if self.grounding is None:
            return concept_targets.new_zeros(())
        return self.grounding.compute_loss(group_feats, concept_targets, self.concept_heads.concept_slices)

    # ── Forward ──────────────────────────────────────────────────────────────

    def forward(
        self,
        input:                  torch.Tensor,
        index_t:                Optional[torch.Tensor] = None,   # khong dung (HyperGCN khong can index_t), giu de tuong thich API voi CLIR
        concept_targets:        Optional[torch.Tensor] = None,
        concept_target_mask:    Optional[torch.Tensor] = None,
        action_labels:          Optional[torch.Tensor] = None,
        return_trace:           bool = False,
    ) -> Union[torch.Tensor, Dict]:
        feature_map = self.backbone.extract_feature_map(input)              # [B, D, T', V=25]

        z_spatial, group_feats = self.concept_heads(feature_map)            # G1
        temporal_feat = self._pool_temporal(feature_map)
        z_temporal = self.temporal_head(temporal_feat)
        z0 = torch.cat([z_spatial, z_temporal], dim=-1)                     # [B, 72]

        reasoner_out = self.reasoner(z0, return_intermediates=True)
        y_final = reasoner_out["y_final"]
        z_final = reasoner_out["z_final"]
        z_all   = reasoner_out["z_all"]
        y_all   = list(reasoner_out["y_all"])
        betas   = reasoner_out["betas"]

        if not self.return_aux and concept_targets is None and not self.training:
            return y_final

        output = {
            "action_logits":          y_final,
            "concept_logits":         z_final,
            "initial_concept_logits": z0,
            "spatial_logits":         z_spatial,
            "temporal_logits":        z_temporal,
            "all_action_logits":      y_all,
            "betas":                  betas,
        }
        if return_trace:
            output["z_all"] = z_all

        if concept_targets is not None and action_labels is not None:
            targets = concept_targets.float().to(z0.device)
            sup_losses = self.compute_sup_loss(z0, z_all, y_all, targets, action_labels, concept_target_mask)
            output.update(sup_losses)

            aux_loss = self.lambda_sup * sup_losses["l_sup"]

            if self.lambda_reg > 0.0:
                reg_losses = self.compute_reg_loss(z_all)
                output.update(reg_losses)
                aux_loss = aux_loss + self.lambda_reg * reg_losses["l_reg"]
            else:
                output["l_reg"] = z0.new_zeros(())

            if self.use_grounding and self.lambda_ground > 0.0:
                l_ground = self.compute_grounding_loss(group_feats, targets)
                output["l_ground"] = l_ground
                aux_loss = aux_loss + self.lambda_ground * l_ground

            output["aux_loss"] = aux_loss

        elif concept_targets is not None:
            targets = concept_targets.float().to(z0.device)
            l_c_K = self._asymmetric_bce(z_final, targets, mask=concept_target_mask)
            l_c_0 = self._asymmetric_bce(z0, targets, mask=concept_target_mask)
            output["l_c_K"] = l_c_K
            output["l_c_0"] = l_c_0
            output["aux_loss"] = l_c_K + 0.5 * l_c_0

        return output


# ─────────────────────────────────────────────────────────────────────────────
# Factory
# ─────────────────────────────────────────────────────────────────────────────

def GLIR_HyperGCN(**kwargs) -> GLIRHyperGCN:
    return GLIRHyperGCN(**kwargs)




## 📜 Block 7: RuleExtractor (DNF Rules & Fidelity Evaluation — kế thừa nguyên trạng từ CLIR)


In [ ]:
# RuleExtractor
"""RuleExtractor: Trích xuất và đánh giá luật DNF sau khi huấn luyện.

Module này thực hiện post-hoc extraction của các luật DNF từ mô hình CLIR đã huấn luyện:
1. Binarize các trọng số theo ngưỡng tau:
   - w_and1 = (sigmoid(theta_and1) >= tau_and)   [L1 AND rules]
   - w_and2 = (sigmoid(theta_and2) >= tau_and2)  [L2a AND2 rules]
   - w_or1  = (sigmoid(theta_or1)  >= tau_or)    [L2b OR1 rules]
   - W_action: lọc các trọng số dương > tau_act (excitatory clauses)

2. Executable DNF Engine:
   - Đánh giá luật dạng boolean/nhị phân trên từng mẫu dữ liệu
   - Kiểm tra L1 conjunction rules (tất cả literal thành phần = 1)
   - Kiểm tra L2a compound rules (tất cả L1 thành phần = 1)
   - Kiểm tra L2b alternative rules (ít nhất 1 L1 thành phần = 1)
   - Tính toán điểm kích hoạt lớp từ các mệnh đề DNF dương

3. Chỉ số đánh giá theo kế hoạch (Mục 10 — CLIR plan):
   - Rule Coverage: Tỷ lệ mẫu được bao phủ bởi ít nhất 1 luật DNF
   - Rule Precision: Tỷ lệ mẫu thỏa mãn luật thực sự thuộc về class đó
   - Model-Rule Fidelity: Tỷ lệ tương đồng giữa dự đoán của mạng nơ-ron và luật logic tượng trưng
"""

from __future__ import annotations
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn

# relative import in notebook scope


_DEFAULT_CONCEPT_NAMES = [f"concept_{i}" for i in range(_DEFAULT_N_CONCEPTS)]


class RuleExtractor:
    """Trình trích xuất và thực thi luật DNF từ CRLReasoner đã huấn luyện.

    Args:
        reasoner:       Instance của CRLReasoner (sau khi load checkpoint).
        concept_names:  Danh sách tên concepts.
        action_names:   Danh sách tên actions.
        tau_and:        Ngưỡng binarize cho L1 AND weights (mặc định 0.5).
        tau_and2:       Ngưỡng binarize cho L2a AND2 weights (mặc định 0.5).
        tau_or:         Ngưỡng binarize cho L2b OR1 weights (mặc định 0.5).
        tau_act:        Ngưỡng trọng số dương ActionHead (mặc định 0.05).
        top_k_l2:       Số L2 rules tối đa cho mỗi action (mặc định 5).
        top_k_l1:       Số L1 rules tối đa trong mỗi L2 rule (mặc định 5).
        top_k_literals: Số literals tối đa trong mỗi L1 rule (mặc định 6).
    """

    def __init__(
        self,
        reasoner,
        concept_names:  Optional[List[str]] = None,
        action_names:   Optional[List[str]] = None,
        tau_and:        float = 0.5,
        tau_and2:       float = 0.5,
        tau_or:         float = 0.5,
        tau_act:        float = 0.05,
        top_k_l2:       int = 5,
        top_k_l1:       int = 5,
        top_k_literals: int = 6,
    ):
        self.reasoner       = reasoner
        self.n_concepts     = reasoner.n_concepts
        self.n_actions      = reasoner.n_actions
        self.concept_names  = concept_names or _DEFAULT_CONCEPT_NAMES[:self.n_concepts]
        self.action_names   = action_names  or [f"action_{i}" for i in range(self.n_actions)]
        self.tau_and        = float(tau_and)
        self.tau_and2       = float(tau_and2)
        self.tau_or         = float(tau_or)
        self.tau_act        = float(tau_act)
        self.top_k_l2       = int(top_k_l2)
        self.top_k_l1       = int(top_k_l1)
        self.top_k_literals = int(top_k_literals)

    # ─── Lấy trọng số binarized (giữ nguyên device hiện tại) ───────────────────

    @torch.no_grad()
    def _get_w_and1(self) -> torch.Tensor:
        """W_and1 binarized [n_literals, r1]."""
        w = torch.sigmoid(self.reasoner.l1_and.weight_logit)
        if self.reasoner.l1_and.group_mask is not None:
            w = w * self.reasoner.l1_and.group_mask
        return (w >= self.tau_and).float()

    @torch.no_grad()
    def _get_w_and2(self) -> torch.Tensor:
        """W_and2 binarized [r1, r2a]."""
        w = torch.sigmoid(self.reasoner.l2a_and2.weight_logit)
        return (w >= self.tau_and2).float()

    @torch.no_grad()
    def _get_w_or1(self) -> torch.Tensor:
        """W_or1 binarized [r1, r2b]."""
        w = torch.sigmoid(self.reasoner.l2b_or1.weight_logit)
        return (w >= self.tau_or).float()

    @torch.no_grad()
    def _get_action_weights(self) -> torch.Tensor:
        """ActionHead weights [r2a + r2b, n_actions]."""
        return self.reasoner.action_head.weight.T

    # ─── Chuyển đổi literal index → tên concept ────────────────────────────

    def _literal_to_name(self, lit_idx: int) -> Tuple[str, bool]:
        n = self.n_concepts
        if lit_idx < n:
            cname = self.concept_names[lit_idx] if lit_idx < len(self.concept_names) else f"c_{lit_idx}"
            return cname, True
        else:
            base_idx = lit_idx - n
            cname = self.concept_names[base_idx] if base_idx < len(self.concept_names) else f"c_{base_idx}"
            return cname, False

    # ─── Trích xuất một L1 rule ─────────────────────────────────────────────

    def _extract_l1_rule(self, w_and1: torch.Tensor, l1_rule_idx: int) -> List[Tuple[str, bool]]:
        active_lits = w_and1[:, l1_rule_idx].nonzero(as_tuple=True)[0].tolist()
        w_col = torch.sigmoid(self.reasoner.l1_and.weight_logit[:, l1_rule_idx])
        if self.reasoner.l1_and.group_mask is not None:
            w_col = w_col * self.reasoner.l1_and.group_mask[:, l1_rule_idx]
        sorted_lits = sorted(active_lits, key=lambda i: -w_col[i].item())
        sorted_lits = sorted_lits[:self.top_k_literals]
        return [self._literal_to_name(i) for i in sorted_lits]

    # ─── Trích xuất rules cho một action ────────────────────────────────────

    def extract_rules_for_action(self, action_idx: int) -> Dict:
        """Trích xuất các mệnh đề DNF khẳng định (w > tau_act) cho action_idx."""
        w_and1 = self._get_w_and1()
        w_and2 = self._get_w_and2()
        w_or1  = self._get_w_or1()
        W_act  = self._get_action_weights()

        r2a = self.reasoner.r2a
        act_weights = W_act[:, action_idx]
        and2_weights = act_weights[:r2a]
        or1_weights  = act_weights[r2a:]

        clauses = []

        # ── AND2 (L2a) — chỉ lấy trọng số dương khẳng định
        top_and2_idx = and2_weights.argsort(descending=True)[:self.top_k_l2].tolist()
        for l2_idx in top_and2_idx:
            w_val = and2_weights[l2_idx].item()
            if w_val < self.tau_act:
                continue
            l1_idx_list = w_and2[:, l2_idx].nonzero(as_tuple=True)[0].tolist()[:self.top_k_l1]
            l1_components = []
            for l1_idx in l1_idx_list:
                lits = self._extract_l1_rule(w_and1, l1_idx)
                if lits:
                    l1_components.append({"l1_rule_idx": l1_idx, "literals": lits})
            if l1_components:
                clauses.append({
                    "type": "and2",
                    "l2_rule_idx": l2_idx,
                    "l1_components": l1_components,
                    "raw_weight": w_val,
                })

        # ── OR1 (L2b) — chỉ lấy trọng số dương khẳng định
        top_or1_idx = or1_weights.argsort(descending=True)[:self.top_k_l2].tolist()
        for l2_idx in top_or1_idx:
            w_val = or1_weights[l2_idx].item()
            if w_val < self.tau_act:
                continue
            l1_idx_list = w_or1[:, l2_idx].nonzero(as_tuple=True)[0].tolist()[:self.top_k_l1]
            l1_components = []
            for l1_idx in l1_idx_list:
                lits = self._extract_l1_rule(w_and1, l1_idx)
                if lits:
                    l1_components.append({"l1_rule_idx": l1_idx, "literals": lits})
            if l1_components:
                clauses.append({
                    "type": "or1",
                    "l2_rule_idx": l2_idx,
                    "l1_components": l1_components,
                    "raw_weight": w_val,
                })

        action_name = self.action_names[action_idx] if action_idx < len(self.action_names) else f"action_{action_idx}"
        return {
            "action_name": action_name,
            "action_idx": action_idx,
            "clauses": clauses,
        }

    def extract_all_rules(self) -> List[Dict]:
        """Trích xuất rules cho tất cả actions và lưu cache."""
        rules = [self.extract_rules_for_action(i) for i in range(self.n_actions)]
        self._cached_rules = rules
        return rules

    def get_or_extract_rules(self) -> List[Dict]:
        """Lấy rules đã trích xuất từ cache hoặc trích xuất mới."""
        if not hasattr(self, "_cached_rules") or self._cached_rules is None:
            self._cached_rules = self.extract_all_rules()
        return self._cached_rules

    # ─── Executable DNF Evaluation Engine ─────────────────────────────────────

    @torch.no_grad()
    def evaluate_dnf(
        self,
        z: torch.Tensor,
        rules: Optional[List[Dict]] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Thực thi chính xác tập luật DNF đã trích xuất (khớp 100% với text/JSON).

        Đánh giá biểu thức boolean của các mệnh đề sau khi đã áp dụng top_k_l2,
        top_k_l1, và top_k_literals, đảm bảo metrics (Coverage, Precision, Fidelity)
        đo lường đúng mô hình DNF thực tế được xuất ra.

        Args:
            z:     Concept logits [B, n_concepts]
            rules: Danh sách rule dicts cho các actions (None = dùng get_or_extract_rules())

        Returns:
            dnf_preds:    [B] predicted class index (-1 nếu không luật nào khớp)
            dnf_scores:   [B, n_actions] activation scores từ các luật DNF
            covered_mask: [B] bool tensor (True nếu có ít nhất 1 luật kích hoạt)
        """
        if rules is None:
            rules = self.get_or_extract_rules()

        device = z.device
        B = z.shape[0]

        # 1. Binarize literals theo ngưỡng z > 0
        hard_pos = (z > 0.0)
        hard_neg = ~hard_pos
        l_bin = torch.cat([hard_pos, hard_neg], dim=-1)  # [B, 2*C] bool tensor

        concept_to_idx = {name: i for i, name in enumerate(self.concept_names)}
        dnf_scores = torch.zeros(B, self.n_actions, device=device)

        # 2. Đánh giá chính xác từng mệnh đề DNF đã trích xuất
        for rule_entry in rules:
            act_idx = rule_entry["action_idx"]
            for clause in rule_entry["clauses"]:
                clause_type = clause["type"]
                w = float(clause["raw_weight"])

                comp_results = []
                for comp in clause["l1_components"]:
                    lits = comp["literals"]
                    if not lits:
                        continue
                    lit_indices = []
                    for cname, is_pos in lits:
                        c_idx = concept_to_idx.get(cname, None)
                        if c_idx is not None and c_idx < self.n_concepts:
                            lit_indices.append(c_idx if is_pos else (c_idx + self.n_concepts))

                    if lit_indices:
                        # L1 rule active iff tất cả literal thành phần thỏa mãn
                        comp_active = l_bin[:, lit_indices].all(dim=-1)  # [B] bool
                        comp_results.append(comp_active)

                if not comp_results:
                    continue

                comp_tensor = torch.stack(comp_results, dim=1)  # [B, num_comps] bool
                if clause_type == "and2":
                    # AND2: Tất cả component L1 rules phải thỏa mãn
                    clause_active = comp_tensor.all(dim=1)  # [B] bool
                elif clause_type == "or1":
                    # OR1: Ít nhất 1 component L1 rule thỏa mãn
                    clause_active = comp_tensor.any(dim=1)  # [B] bool
                else:
                    clause_active = comp_tensor.all(dim=1)

                dnf_scores[:, act_idx] += w * clause_active.float()

        # 3. Mẫu được bao phủ nếu có ít nhất 1 class nhận score > 0
        max_scores, best_class = dnf_scores.max(dim=-1)
        covered_mask = max_scores > 0.0

        dnf_preds = torch.where(covered_mask, best_class, torch.tensor(-1, device=device))

        return dnf_preds, dnf_scores, covered_mask

    # ─── Đánh giá Fidelity, Coverage, Precision theo chuẩn kế hoạch ───────────

    @torch.no_grad()
    def evaluate_fidelity(
        self,
        model,
        data_loader,
        device: str = "cpu",
        n_batches: int = 50,
    ) -> Dict[str, float]:
        """Tính toán: Rule Precision, Rule Coverage, và Model-Rule Fidelity.

        Returns:
            {
                "coverage": float,   # Tỷ lệ mẫu được ít nhất 1 luật DNF bao phủ
                "precision": float,  # Độ chính xác của các dự đoán luật DNF (trên tập covered)
                "fidelity": float,   # Tỷ lệ dự đoán NN trùng khớp với luật DNF executable
                "total_samples": int,
                "covered_samples": int,
            }
        """
        model.eval()
        rules = self.get_or_extract_rules()
        total_samples = 0
        covered_samples = 0
        rule_correct = 0
        model_rule_agree = 0

        for batch_idx, batch in enumerate(data_loader):
            if batch_idx >= n_batches:
                break
            if len(batch) >= 5:
                data, index_t, label, index, _ = batch[:5]
            else:
                data, index_t, label, index = batch[:4]

            data    = data.float().to(device)
            index_t = index_t.float().to(device)
            label   = label.long().to(device)

            output = model(data, index_t=index_t)
            if isinstance(output, dict):
                nn_preds = output["action_logits"].argmax(dim=-1)
                z_final = output["concept_logits"]
            else:
                nn_preds = output.argmax(dim=-1)
                z_final = None

            if z_final is None:
                continue

            # Đánh giá chính xác tập luật DNF đã trích xuất
            dnf_preds, _, covered_mask = self.evaluate_dnf(z_final, rules=rules)

            b_total = len(label)
            total_samples += b_total

            # Coverage
            b_cov = covered_mask.sum().item()
            covered_samples += b_cov

            # Precision (trên các mẫu được bao phủ)
            if b_cov > 0:
                rule_correct += (dnf_preds[covered_mask] == label[covered_mask]).sum().item()

            # Fidelity: NN prediction trùng với DNF prediction
            model_rule_agree += (nn_preds == dnf_preds).sum().item()

        coverage = (covered_samples / total_samples) if total_samples > 0 else 0.0
        precision = (rule_correct / covered_samples) if covered_samples > 0 else 0.0
        fidelity = (model_rule_agree / total_samples) if total_samples > 0 else 0.0

        return {
            "coverage": coverage,
            "precision": precision,
            "fidelity": fidelity,
            "total_samples": total_samples,
            "covered_samples": covered_samples,
        }

    # ─── Format DNF dạng văn bản ─────────────────────────────────────────────

    @staticmethod
    def format_rule_as_dnf(rule: Dict) -> str:
        action_name = rule["action_name"]
        lines = [f"Action '{action_name}' (idx={rule['action_idx']}):"]

        for ci, clause in enumerate(rule["clauses"]):
            clause_type = clause["type"].upper()
            l2_idx = clause["l2_rule_idx"]
            w = clause["raw_weight"]
            parts = []
            for comp in clause["l1_components"]:
                lits = comp["literals"]
                lit_strs = [
                    f"{'NOT ' if not pos else ''}{name}"
                    for name, pos in lits
                ]
                parts.append(f"({' ∧ '.join(lit_strs)})")
            connector = " ∨ " if clause_type == "OR1" else " ∧ "
            clause_str = f"  Clause {ci+1} [{clause_type}, L2={l2_idx}, w={w:.3f}]:\n    {connector.join(parts)}"
            lines.append(clause_str)

        if not rule["clauses"]:
            lines.append("  (No active affirmative rules extracted)")

        return "\n".join(lines)

    @staticmethod
    def save_rules(rules: List[Dict], output_dir: str):
        """Lưu rules trích xuất ra file JSON và text trong output_dir."""
        import os
        import json
        os.makedirs(output_dir, exist_ok=True)
        json_path = os.path.join(output_dir, "dnf_rules.json")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(rules, f, ensure_ascii=False, indent=2)

        txt_path = os.path.join(output_dir, "dnf_rules.txt")
        with open(txt_path, "w", encoding="utf-8") as f:
            for rule in rules:
                f.write(RuleExtractor.format_rule_as_dnf(rule) + "\n\n")




## 🧪 Block 8: Kiểm thử nhanh tính toàn vẹn (Quick Smoke Test — V2 và V3, 60CSub)


In [ ]:
# Quick Model Verification Test cho NTU60 Cross-Subject (60 classes)
model_smoke = GLIRHyperGCN(
    num_classes=60,
    num_concepts=72,
    spatial_concepts=52,
    temporal_concepts=20,
    num_point=25,           # NTU-25 chuan (KHONG phai 24 nhu SkateFormer)
    embedding_channels=128,
    num_loops=2,
    lambda_sup=1.0,
    lambda_reg=0.0,
    use_grounding=True,
)
print("GLIR-HyperGCN (60CSub) initialized successfully!")
print(f"Total Parameters: {sum(p.numel() for p in model_smoke.parameters() if p.requires_grad):,}")
print(f"Grounding available (CLIP loaded): {model_smoke.grounding.available if model_smoke.grounding else False}")

# Test Dummy Forward — V=25 khop, khong phai V=24 nhu SkateFormer
x_dummy = torch.randn(2, 3, 64, 25, 2)
c_targets = torch.randint(0, 2, (2, 72)).float()
a_labels = torch.randint(0, 60, (2,)).long()

# V2 — set_phase() roi rac
model_smoke.set_phase(1)
out1 = model_smoke(x_dummy, concept_targets=c_targets, action_labels=a_labels)
print(f"[V2] Phase 1 Loss: {out1['aux_loss'].item():.4f}, beta={model_smoke.reasoner.get_betas().detach().numpy()}")
model_smoke.set_phase(2, temperature=6.0)
out2 = model_smoke(x_dummy, concept_targets=c_targets, action_labels=a_labels)
print(f"[V2] Phase 2 Loss: {out2['aux_loss'].item():.4f}, l_reg: {out2['l_reg'].item():.4f}")
assert out2["action_logits"].shape == (2, 60)

# V3 — set_progress() lien tuc
for progress in (0.0, 0.5, 1.0):
    model_smoke.set_progress(progress)
    out3 = model_smoke(x_dummy, concept_targets=c_targets, action_labels=a_labels)
    print(f"[V3] progress={progress:.1f}: temperature={model_smoke.reasoner.temperature:.2f}, "
          f"beta_max={model_smoke.reasoner.beta_max:.2f}, loss={out3['aux_loss'].item():.4f}")
assert out3["action_logits"].shape == (2, 60)
print("Smoke Test (V2 + V3) for 60CSub Passed!")



## ⚙️ Block 9: Cấu hình Huấn luyện, chọn VARIANT (V2/V3) & Chuẩn bị DataLoaders
Mặc định **NTU60 Cross-Subject (60CSub)**. `VARIANT` quyết định lịch huấn luyện dùng ở Block 10:
`v2` = `set_phase()` rời rạc (giống CLIR gốc, dùng làm baseline so sánh); `v3` = `set_progress()` liên tục (đề xuất chính).



In [ ]:
# Cấu hình Benchmark & Variant
BENCHMARK = "ntu60_cs"  # @param ["ntu60_cs", "ntu60_cv", "ntu120_csub", "ntu120_cset"]
VARIANT = "v3"           # @param ["v2", "v3"]

DATASET_CONFIGS = {
    "ntu60_cs": {
        "num_classes": 60,
        "train_data": "/content/drive/MyDrive/KLTN/KLTN_Huy-Nhu/NTU60_CS_64f.npz",
        "test_data": "/content/drive/MyDrive/KLTN/KLTN_Huy-Nhu/NTU60_CS_64f.npz",
        "backbone_weights": "/content/ntu60_CS/HyperGCN_j.pt",
        "work_dir": "./work_dir/ntu60_cs/glir_hypergcn",
    },
    "ntu60_cv": {
        "num_classes": 60,
        "train_data": "/content/drive/MyDrive/KLTN/KLTN_Huy-Nhu/NTU60_CV_64f.npz",
        "test_data": "/content/drive/MyDrive/KLTN/KLTN_Huy-Nhu/NTU60_CV_64f.npz",
        "backbone_weights": "/content/ntu60_CV/HyperGCN_j.pt",
        "work_dir": "./work_dir/ntu60_cv/glir_hypergcn",
    },
    "ntu120_csub": {
        "num_classes": 120,
        "train_data": "/content/drive/MyDrive/KLTN/KLTN_Huy-Nhu/NTU120_CSub_64f.npz",
        "test_data": "/content/drive/MyDrive/KLTN/KLTN_Huy-Nhu/NTU120_CSub_64f.npz",
        "backbone_weights": "/content/ntu120_CSub/HyperGCN_j.pt",
        "work_dir": "./work_dir/ntu120_csub/glir_hypergcn",
    },
    "ntu120_cset": {
        "num_classes": 120,
        "train_data": "/content/drive/MyDrive/KLTN/KLTN_Huy-Nhu/NTU120_CSet_64f.npz",
        "test_data": "/content/drive/MyDrive/KLTN/KLTN_Huy-Nhu/NTU120_CSet_64f.npz",
        "backbone_weights": "/content/ntu120_CSet/HyperGCN_j.pt",
        "work_dir": "./work_dir/ntu120_cset/glir_hypergcn",
    },
}

selected = DATASET_CONFIGS[BENCHMARK]
work_dir = selected["work_dir"] + f"_{VARIANT}"

TRAIN_CONFIG = {
    "benchmark": BENCHMARK,
    "variant": VARIANT,
    "num_classes": selected["num_classes"],
    "num_concepts": 72,
    "spatial_concepts": 52,
    "temporal_concepts": 20,
    "num_point": 25,
    "embedding_channels": 128,

    "train_data": selected["train_data"],
    "test_data": selected["test_data"],
    "backbone_weights": selected["backbone_weights"],
    "work_dir": work_dir,

    "batch_size": 16,
    "test_batch_size": 16,
    "num_epoch": 70,
    "phase1_epochs": 30,     # chi dung cho VARIANT == "v2"
    "base_lr": 1e-4,
    "min_lr": 1e-6,
    "warmup_lr": 1e-6,
    "warm_up_epoch": 10,
    "weight_decay": 0.05,
    "grad_clip": 1.0,

    "lambda_sup": 1.0,
    "lambda_reg": 0.005,
    "lambda_ground": 0.1,

    "num_loops": 2,
    "beta_init": 0.1,
    "beta_max": 0.5,
    "use_grounding": True,
}

os.makedirs(TRAIN_CONFIG["work_dir"], exist_ok=True)
print(f"Active Benchmark: {BENCHMARK.upper()} ({TRAIN_CONFIG['num_classes']} action classes) — VARIANT={VARIANT.upper()}")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k}: {v}")



In [ ]:
# Khởi tạo DataLoader nếu file dữ liệu tồn tại
from torch.utils.data import DataLoader

if os.path.exists(TRAIN_CONFIG["train_data"]):
    print(f"Loading {TRAIN_CONFIG['benchmark'].upper()} dataset from disk...")
    train_dataset = Feeder(
        data_path=TRAIN_CONFIG["train_data"],
        concepts_csv="concepts/ntu_rgbd_60_120.csv",
        split="train",
        window_size=64,
        thres=64,
        uniform=True,
        partition=True,
        remove_null_concept=True,
        p_interval=[0.5, 1],
        aug_method="a123489",
        intra_p=0.5,
        inter_p=0.2,
    )
    test_dataset = Feeder(
        data_path=TRAIN_CONFIG["test_data"],
        concepts_csv="concepts/ntu_rgbd_60_120.csv",
        split="test",
        window_size=64,
        thres=64,
        uniform=True,
        partition=True,
        remove_null_concept=True,
        p_interval=[0.95],
    )
    train_loader = DataLoader(train_dataset, batch_size=TRAIN_CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=TRAIN_CONFIG["test_batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    print(f"DataLoaders ready! Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")
else:
    print(f"Warning: {TRAIN_CONFIG['train_data']} not found. DataLoaders skipped.")
    train_loader, test_loader = None, None



## 🏋️ Block 10: Vòng lặp Huấn luyện (GLIRTrainer — hỗ trợ cả V2 và V3) & Lưu Checkpoint
- **V2** (`set_phase`): Phase 1 (Epoch 1–30, `soft`, `beta_max=0.1`) → Phase 2 (Epoch 31+, `ste`, `beta_max=0.5`, `lambda_reg=0.005`) — **đột ngột**.
- **V3** (`set_progress`): nội suy liên tục `temperature: 1→10`, `beta_max: 0.1→0.5`, `lambda_reg: 0→0.005` theo `epoch/num_epoch` — **không có bước nhảy**.
- `checkpoint_latest.pt` lưu mỗi epoch (đầy đủ optimizer để resume); `best_checkpoint.pt` lưu đầy đủ khi đạt best acc, `best_model.pt` chỉ lưu weights (dùng cho RuleExtractor/inference).



In [ ]:
from tqdm import tqdm
import torch.optim as optim
from timm.scheduler.cosine_lr import CosineLRScheduler

class GLIRTrainer:
    def __init__(self, cfg, train_loader, test_loader):
        self.cfg = cfg
        self.variant = cfg["variant"]
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.model = GLIRHyperGCN(
            num_classes=cfg["num_classes"],
            num_concepts=cfg["num_concepts"],
            spatial_concepts=cfg["spatial_concepts"],
            temporal_concepts=cfg["temporal_concepts"],
            num_point=cfg["num_point"],
            embedding_channels=cfg["embedding_channels"],
            backbone_weights=cfg["backbone_weights"] if os.path.exists(cfg["backbone_weights"]) else None,
            num_loops=cfg["num_loops"],
            beta_max=cfg["beta_max"],
            beta_init=cfg["beta_init"],
            lambda_sup=cfg["lambda_sup"],
            lambda_reg=cfg["lambda_reg"] if self.variant == "v3" else 0.0,
            lambda_ground=cfg["lambda_ground"],
            use_grounding=cfg["use_grounding"],
        ).to(self.device)

        self.best_acc = 0.0
        self.global_step = 0

        if self.variant == "v2":
            self.model.set_phase(1)
        else:
            self.model.set_progress(0.0)

        self.optimizer = optim.AdamW(self.model.parameters(), lr=cfg["base_lr"], weight_decay=cfg["weight_decay"])
        if self.train_loader is not None:
            n_iter = len(self.train_loader)
            total_steps = cfg["num_epoch"] * n_iter
            warmup_steps = cfg["warm_up_epoch"] * n_iter
            self.scheduler = CosineLRScheduler(
                self.optimizer, t_initial=total_steps, lr_min=cfg["min_lr"],
                warmup_lr_init=cfg["warmup_lr"], warmup_t=warmup_steps, cycle_limit=1, t_in_epochs=False,
            )

    def _set_epoch_schedule(self, epoch):
        num_epoch = self.cfg["num_epoch"]
        if self.variant == "v2":
            new_phase = 1 if epoch < self.cfg["phase1_epochs"] else 2
            want_mode = "soft" if new_phase == 1 else "ste"
            if self.model.reasoner.literal_mode != want_mode:
                print(f"\n{'='*60}\n[V2][PHASE SWITCH] -> Phase {new_phase} tai epoch {epoch+1} (dot ngot)\n{'='*60}")
                self.model.set_phase(new_phase)
        else:
            progress = epoch / max(1, num_epoch - 1)
            self.model.set_progress(progress)

    def train_epoch(self, epoch):
        self.model.train()
        self._set_epoch_schedule(epoch)
        total_loss, correct, total = 0.0, 0, 0

        pbar = tqdm(self.train_loader, desc=f"[{self.variant.upper()}] Epoch {epoch+1}/{self.cfg['num_epoch']}")
        for batch in pbar:
            if len(batch) == 5:
                data, _index_t, label, _, c_targets = batch
            else:
                data, _index_t, label, _ = batch[:4]
                c_targets = None

            data = data.float().to(self.device)
            label = label.long().to(self.device)
            c_targets = c_targets.float().to(self.device) if c_targets is not None else None

            self.scheduler.step_update(self.global_step)
            self.global_step += 1

            self.optimizer.zero_grad()
            out = self.model(data, concept_targets=c_targets, action_labels=label)
            loss = torch.nn.functional.cross_entropy(out["action_logits"], label)
            if "aux_loss" in out:
                loss += out["aux_loss"]
            loss.backward()
            if self.cfg["grad_clip"] > 0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg["grad_clip"])
            self.optimizer.step()

            total_loss += loss.item()
            pred = out["action_logits"].argmax(dim=-1)
            correct += (pred == label).sum().item()
            total += label.size(0)
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{correct/total*100:.2f}%"})

        return total_loss / len(self.train_loader), correct / total

    @torch.no_grad()
    def eval_epoch(self):
        self.model.eval()
        correct, total = 0, 0
        for batch in tqdm(self.test_loader, desc="Evaluation"):
            data, label = batch[0].float().to(self.device), batch[2].long().to(self.device)
            out = self.model(data)
            logits = out["action_logits"] if isinstance(out, dict) else out
            pred = logits.argmax(dim=-1)
            correct += (pred == label).sum().item()
            total += label.size(0)
        return correct / total if total > 0 else 0.0

    def _build_checkpoint(self, epoch):
        return {
            "epoch": epoch + 1,
            "global_step": self.global_step,
            "variant": self.variant,
            "best_acc": self.best_acc,
            "model": self.model.state_dict(),
            "optimizer": self.optimizer.state_dict(),
        }

    def run(self):
        print(f"Starting full training ({self.variant.upper()}) for {self.cfg['num_epoch']} epochs on {self.cfg['benchmark'].upper()}...")
        for epoch in range(self.cfg["num_epoch"]):
            train_loss, train_acc = self.train_epoch(epoch)

            latest_path = os.path.join(self.cfg["work_dir"], "checkpoint_latest.pt")
            torch.save(self._build_checkpoint(epoch), latest_path)

            if (epoch + 1) % 5 == 0 or (epoch + 1) == self.cfg["num_epoch"]:
                val_acc = self.eval_epoch()
                print(f"[Epoch {epoch+1}] Train Loss: {train_loss:.4f}, Train Acc: {train_acc*100:.2f}%, Val Acc: {val_acc*100:.2f}%")

                if val_acc > self.best_acc:
                    self.best_acc = val_acc
                    save_path = os.path.join(self.cfg["work_dir"], "best_model.pt")
                    torch.save(self.model.state_dict(), save_path)
                    best_ckpt_path = os.path.join(self.cfg["work_dir"], "best_checkpoint.pt")
                    torch.save(self._build_checkpoint(epoch), best_ckpt_path)
                    print(f"★ Saved new best model to {save_path} (Acc: {self.best_acc*100:.2f}%)")

print("GLIRTrainer defined successfully!")



In [ ]:
# Chạy huấn luyện (Bỏ comment để bắt đầu huấn luyện nếu đã có dữ liệu)
# if train_loader is not None and test_loader is not None:
#     trainer = GLIRTrainer(TRAIN_CONFIG, train_loader, test_loader)
#     trainer.run()



## 📜 Block 11: Trích xuất Luật Logic DNF & Đánh giá Tính giải thích (Rule Extraction)
Sau khi có checkpoint đã huấn luyện (hoặc dùng `model_smoke` để thử nghiệm), trích xuất các mệnh đề DNF tường minh cho 60 hành động NTU60 và đánh giá **Rule Coverage**, **Rule Precision**, **Model-Rule Fidelity**.



In [ ]:
# Đọc danh sách tên concept và action (khớp với số lớp của benchmark)
concept_names, action_names = [], []
with open("concepts/ntu_rgbd_60_120.csv", "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f if l.strip()]
    concept_names = [h.strip() for h in lines[0].split(",")[1:]]
    action_names = [l.split(",")[0].strip() for l in lines[1:]]

action_names = action_names[:TRAIN_CONFIG["num_classes"]]
print(f"Loaded {len(concept_names)} concepts and {len(action_names)} actions ({TRAIN_CONFIG['benchmark'].upper()}).")

# Dùng model_smoke (Block 8) hoặc nạp lại từ best_model.pt sau khi train
model_for_extract = model_smoke.reasoner.crl
model_smoke.reasoner.literal_mode = "ste"  # bat buoc STE de trich xuat luat

extractor = RuleExtractor(
    reasoner=model_for_extract,
    concept_names=concept_names,
    action_names=action_names,
    top_k_l2=3,
    top_k_l1=3,
    top_k_literals=4,
)

os.makedirs("rules_output", exist_ok=True)
all_rules = extractor.extract_all_rules()

print(f"Successfully extracted DNF rules for all {len(all_rules)} actions!\n")
print("--- Ví dụ 3 luật DNF trích xuất mẫu ---")
for act_idx in [0, 1, 2]:
    text_rule = RuleExtractor.format_rule_as_dnf(all_rules[act_idx])
    print(text_rule)
    print("-" * 50)

RuleExtractor.save_rules(all_rules, "rules_output")
print("Saved rules to: rules_output/dnf_rules.json and rules_output/dnf_rules.txt")



In [ ]:
# Đánh giá Fidelity, Coverage, Precision trên tập kiểm tra thực tế
if test_loader is not None:
    print("Evaluating DNF rule metrics on test set...")
    metrics = extractor.evaluate_fidelity(model_smoke, test_loader, device="cuda" if torch.cuda.is_available() else "cpu", n_batches=30)
    print("\n" + "="*50)
    print("🎯 DNF RULE EXPLAINABILITY EVALUATION")
    print("="*50)
    print(f"  • Rule Coverage:       {metrics['coverage']*100:.2f}%  (Tỷ lệ mẫu có ít nhất 1 luật kích hoạt)")
    print(f"  • Rule Precision:      {metrics['precision']*100:.2f}%  (Độ chính xác khi luật DNF đưa ra dự đoán)")
    print(f"  • Model-Rule Fidelity: {metrics['fidelity']*100:.2f}%  (Độ tương đồng giữa luật và mạng nơ-ron)")
    print("="*50)
else:
    print("Test loader not active. DNF evaluation on dummy concept logits:")
    z_eval = torch.randn(8, 72)
    preds, scores, cov = extractor.evaluate_dnf(z_eval)
    print(f"Evaluated on 8 samples. Covered: {cov.sum().item()}/8")



## 💻 Block 12: Huấn luyện & Trích xuất qua Dòng lệnh CLI (NTU60 Cross-Subject)
Nếu bạn muốn huấn luyện theo script chuẩn hóa qua terminal hoặc cell Colab (cần clone repo `GLIR-HyperGCN` đầy đủ, không chỉ notebook này):



In [ ]:
# 1. Huấn luyện qua dòng lệnh — VARIANT v2 (set_phase rời rạc, baseline) hoặc v3 (set_progress liên tục, đề xuất)
!python train.py --variant v3 --config config/ntu60_cs/glir_hypergcn.yaml
# !python train.py --variant v2 --config config/ntu60_cs/glir_hypergcn.yaml

# 2. Trích xuất luật DNF sau huấn luyện
# !python scripts/extract_rules.py \
#     --checkpoint work_dir/ntu60_cs/glir_hypergcn_v3/best_model.pt \
#     --config config/ntu60_cs/glir_hypergcn.yaml \
#     --output-dir work_dir/rules_ntu60_cs \
#     --eval-fidelity

